[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/tokenization-embeddings-course/20_capstone/20_capstone.ipynb)

# Módulo 20 · Capstone: Políglota completo

**Nivel:** 🔴 Experto · **Duración:** ≈ 6 h (lección ≈ 2.5 h + proyecto ≈ 3.5 h) · **GPU:** no necesaria en `FAST_DEV_RUN` (CPU, ≈ 2 min); T4/L4 para modelos reales y el corpus completo (≈ 1–3 unidades de Colab) · **Prerrequisitos:** módulos 00–19.

> 🧭 **Ruta núcleo vs ampliación.** Núcleo (lo que usa el proyecto): §1 criterios de aceptación → §2–§5 decisiones por componente → §6 servicio (TODO 4 del proyecto) → §7 evaluación en test ciego → §8 informe. Ampliación: la simulación de carga de §6 (puedes ejecutarla sin estudiarla) y §9 multimodal opcional.

**Hilo conductor — Políglota v1.0.** Has construido cada pieza por separado: tokenizer (02–05), embedder (09–12), híbrido (13), índice (14), evaluación (15), multimodal (16), interpretabilidad (17), servicio (18) y criterios de diseño (19). Esta lección es la **solución de referencia del sistema completo**: integra las piezas en un pipeline, **decide cada componente con datos de dev**, evalúa una sola vez en un **test ciego** contra el baseline BM25 del módulo 00 y genera automáticamente el **informe técnico** con métricas, costes, latencia y criterios de aceptación. El proyecto te pide hacer lo mismo con tus propias decisiones.

> **Estado del arte a octubre de 2026.** Las piezas usadas son 🟢/🟡 (ver mapa de madurez). Las cifras de este notebook salen del corpus **sintético** + fallbacks si no hay acceso a Hugging Face: el *procedimiento* es el entregable; los números, de tu ejecución.

## 🎯 Objetivos de aprendizaje
1. **Definir** criterios de aceptación medibles *antes* de construir (calidad, latencia, memoria, coste).
2. **Integrar** tokenizer, analizador léxico, embedder, fusión híbrida, índice ANN y servicio en un pipeline reproducible.
3. **Decidir** cada componente en un split de desarrollo con reglas explícitas, sin tocar el test.
4. **Evaluar** una única vez en test ciego con IC pareados y desglose por idioma contra el baseline del módulo 00.
5. **Proyectar** memoria, coste y latencia a la escala de producción (10 M de chunks).
6. **Redactar** un informe técnico de decisiones (ADR) generado desde las métricas y verificar los criterios de aceptación.

## 🔁 Conexión con módulos anteriores
<details><summary><b>1 · (M15/M19)</b> ¿Por qué separamos dev y test ciego antes de empezar?</summary>

Porque todas las decisiones (vocabulario, dimensión, fusión, índice) son selecciones entre alternativas: si se eligen mirando el test, el número final es optimista (T1/T2 del módulo 19). El test se usa **una vez**, al final.
</details>
<details><summary><b>2 · (M04)</b> ¿Qué métricas reportamos siempre de un tokenizer?</summary>

Fertilidad (tokens/palabra) y bytes por token, por idioma, y la prima de tokens respecto al inglés.
</details>
<details><summary><b>3 · (M13)</b> ¿Por qué RRF es una buena opción por defecto para fusionar?</summary>

Usa solo rangos: no necesita calibrar scores de escalas distintas y no tiene pesos que ajustar (salvo $k$), así que no hay fuga de tuning. La combinación lineal puede ganar, pero hay que ajustarla con validación cruzada.
</details>
<details><summary><b>4 · (M18)</b> ¿Qué tres cosas deben ir en la «versión» de un vector?</summary>

Modelo/checkpoint, preprocesado (normalización, truncado, prefijo, dimensión y precisión) y servidor. Va en la clave de caché y en los metadatos del índice.
</details>
<details><summary><b>5 · (M00, diagnóstico)</b> ¿Por qué el listón del capstone es el BM25 del módulo 00 y no «el mejor embedder de MTEB»?</summary>

Porque es el sistema que ya funciona con **tus** datos, sin GPU y sin entrenar: un sistema nuevo que no le gana en tu golden set (con IC pareado y por idioma) no justifica su coste. BEIR mostró que muchos densos zero-shot pierden contra BM25 fuera de dominio (módulos 00 y 15). Y las métricas son las mismas del 00 (nDCG@10 con ganancia lineal, IC bootstrap por queries): cambiarlas a mitad de curso haría incomparables los números.
</details>
<details><summary><b>6 · (M10 + M12, cálculo)</b> $v = (0.6, 0.8, 0, 0)$ y $w = (0.6, 0, 0.8, 0)$ son unitarios. Recorta ambos a 2 dimensiones. ¿Qué coseno da el producto escalar sin renormalizar y cuál tras renormalizar?</summary>

Recortados: $v' = (0.6, 0.8)$, $w' = (0.6, 0)$. Sin renormalizar, $v'\cdot w' = 0.36$ (no es un coseno: $\lVert w'\rVert = 0.6$). Renormalizando: $w'' = (1, 0)$, coseno $= 0.6$. Por eso `EmbedderRecortado` aplica `l2n` tras recortar (Matryoshka, módulo 12; normalización y anisotropía, módulo 10).
</details>

## 🧩 Qué integra este capstone (y qué queda como reto)
Antes de construir, sitúa cada módulo. La columna «dónde» es tu índice de repaso: si una sección te resulta opaca, vuelve al módulo de la izquierda.

| Módulo | Qué aporta a Políglota v1.0 | Dónde |
|---|---|---|
| 00 | corpus, golden set, nDCG@10, BM25 baseline, IC bootstrap | §0, §2, §7 |
| 01 | NFC en indexación y consulta (y en la clave de caché) | §2, §6 |
| 02–04 | BPE byte-level, fertilidad, bytes/token y prima de tokens → coste | §2, §8 |
| 05 | robustez ante Unicode invisible y tokens raros | reto 6 del proyecto |
| 06–08 | tokenizer-free y tokens multimodales: 🔴/🟡, fuera de la v1.0 | §9 (opcional) |
| 07, 09 | el embedder trae su tokenizer: no se cambia sin *tokenizer transfer* | §2 (👀) |
| 10–12 | bi-encoder, normalización, Matryoshka (dimensión) e int8 | §3 |
| 11 | tu embedder contrastivo propio como candidato | reto 1 del proyecto |
| 13 | BM25 + denso con RRF o lineal (elegida con CV) | §4 |
| 14 | Flat vs HNSW, neighbor recall, p99 | §5 |
| 15 | dev / test ciego estratificado, test pareado, slices | §1, §3, §7 |
| 16 | retriever visual en 2ª etapa | §9, reto 3 |
| 17 | los vectores son datos sensibles (inversión) | reto 7 del proyecto |
| 18 | caché versionada, alias, servicio, latencia | §6 |
| 19 | árbol del índice, trampas T1–T2, regla «lo simple salvo prueba» | §4, §5, §7 |

> ### 🎨 GEMINI-IMAGEN · ID: `M20-IMG-01` · ⚠️ REEMPLAZAR ESTA CELDA
> **Arquitectura de Políglota v1.0**
>
> **Qué muestra:** El sistema completo que se integra en el capstone, con cada caja etiquetada con el módulo del curso que la construyó.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Arquitectura de Políglota v1.0 con la ruta de indexación arriba, la de consulta abajo y la evaluación e informe a la derecha; cada caja indica su módulo.
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama de arquitectura técnico, fondo blanco, paleta sobria: azul #1F6FEB (ruta de consulta), naranja #F28E2B (ruta de indexación), verde #2CA02C (evaluación e informe), gris #6B7280 (almacenamiento). Sans-serif legible, flechas claras, cajas con esquinas suaves.
> Arriba, de izquierda a derecha (naranja), título "Indexación": "Docs ES/EN + código" → "Normalización NFC (M01)" → "Chunking" → "Tokenizer BPE: coste y fertilidad (M02–M04)" → bifurcación en dos cajas: "BM25 · palabras + identificadores (M00, M13)" y "Embedder · dimensión recortada + int8 (M10–M12)" → cilindros grises "Índice invertido" y "Índice vectorial Flat/HNSW (M14)".
> Abajo, de izquierda a derecha (azul), título "Consulta": "Usuario" → "API FastAPI /search (M18)" → "Caché LRU versionada (M18)" → flechas a los dos índices → "Fusión RRF (M13)" → "Top-10".
> A la derecha, una columna verde: "Evaluación: dev para decidir · test ciego una vez · IC pareado · slices (M15, M19)" → "Informe técnico + criterios de aceptación (M20)".
> Una caja punteada gris opcional junto al embedder: "Opcional: páginas/capturas → ColPali MaxSim en 2ª etapa (M16)".
> Evita: logotipos, texto pequeño, iconos decorativos, cifras de rendimiento.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M20-IMG-01.png` en la carpeta del módulo y reemplaza esta celda por: `![Arquitectura de Políglota v1.0 con la ruta de indexación arriba, la de consulta abajo y la evaluación e informe a la derecha; cada caja indica su módulo.](assets/M20-IMG-01.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> INDEXACIÓN: Docs → NFC(M01) → chunks → BPE(M02-04) → [BM25 palabras+ident (M00,M13)] [Embedder d recortada+int8 (M10-12)] → índices (M14)
> CONSULTA  : Usuario → FastAPI(M18) → Caché versionada(M18) → BM25 + denso → RRF(M13) → top-10
> EVALUACIÓN: dev decide · test ciego 1 vez · IC pareado · slices (M15,M19) → INFORME + criterios (M20)
> ```

> 👀 **Qué debes observar:** no hay ninguna caja nueva: el capstone es **integración**. Lo nuevo es la columna verde: cada decisión de la izquierda se toma con datos de *dev* y se justifica en el informe.

## 0. Preparación
Piezas canónicas del curso, copiadas para que el notebook sea autocontenido: corpus, métricas y BM25 (módulo 00), estadística pareada (15), embedders con fallback (12), caché y servicio versionado (18).

In [ ]:
!pip install -q faiss-cpu qdrant-client fastapi httpx sentence-transformers scikit-learn matplotlib pandas tokenizers

In [ ]:
import os, sys, time, math, random, json, hashlib, unicodedata, warnings
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED)
FAST_DEV_RUN = True       # True: tamaños diminutos (CPU en minutos). False en Colab con GPU para modelos reales y más datos.
N_THREADS = int(os.environ.get("N_THREADS", "1" if FAST_DEV_RUN else "4"))
os.environ.setdefault("OMP_NUM_THREADS", str(N_THREADS))
try:
    import torch
    torch.manual_seed(SEED); torch.set_num_threads(N_THREADS)
    device = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    torch, device = None, "cpu"
AZUL, NARANJA, VERDE, GRIS, ROJO = "#1F6FEB", "#F28E2B", "#2CA02C", "#6B7280", "#D62728"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .25})
print(f"device={device} · FAST_DEV_RUN={FAST_DEV_RUN} · threads={N_THREADS}")

In [ ]:
# ── Datos para el corpus SINTÉTICO de Políglota (último recurso si no hay red / Hugging Face) ──
# (artículo+concepto ES, artículo+concepto EN, definición ES, definición EN)
CONCEPTOS = [
    ("el índice invertido", "the inverted index", "una estructura que asocia cada término con la lista de documentos que lo contienen", "a structure that maps each term to the list of documents that contain it"),
    ("la caché", "the cache", "una memoria rápida que guarda resultados recientes para no recalcularlos", "a fast memory that stores recent results so they are not recomputed"),
    ("la base de datos relacional", "the relational database", "un sistema que organiza los datos en tablas relacionadas mediante claves", "a system that organizes data into tables related through keys"),
    ("el balanceador de carga", "the load balancer", "un componente que reparte las peticiones entre varios servidores", "a component that spreads requests across several servers"),
    ("el contenedor Docker", "the Docker container", "un proceso aislado que empaqueta una aplicación con sus dependencias", "an isolated process that packages an application with its dependencies"),
    ("el control de versiones", "version control", "un sistema que registra los cambios de los archivos a lo largo del tiempo", "a system that records changes to files over time"),
    ("la API REST", "the REST API", "una interfaz que expone recursos mediante URL y verbos HTTP", "an interface that exposes resources through URLs and HTTP verbs"),
    ("la autenticación con tokens", "token authentication", "un mecanismo que identifica al usuario con un testigo firmado en cada petición", "a mechanism that identifies the user with a signed token on every request"),
    ("la expresión regular", "the regular expression", "un patrón que describe un conjunto de cadenas de texto", "a pattern that describes a set of text strings"),
    ("el hilo de ejecución", "the thread", "una secuencia de instrucciones que comparte memoria con otros hilos del proceso", "a sequence of instructions that shares memory with other threads of the process"),
    ("el recolector de basura", "the garbage collector", "un mecanismo que libera automáticamente la memoria que ya no se usa", "a mechanism that automatically frees memory that is no longer used"),
    ("la cola de mensajes", "the message queue", "un búfer que desacopla a los productores de los consumidores de eventos", "a buffer that decouples event producers from consumers"),
    ("la tabla hash", "the hash table", "una estructura que guarda pares clave-valor y busca en tiempo casi constante", "a structure that stores key-value pairs and looks them up in near-constant time"),
    ("el árbol binario de búsqueda", "the binary search tree", "un árbol ordenado donde cada nodo separa los valores menores de los mayores", "an ordered tree where each node separates smaller values from larger ones"),
    ("la red neuronal", "the neural network", "un modelo formado por capas de unidades que aprenden pesos a partir de ejemplos", "a model made of layers of units that learn weights from examples"),
    ("el descenso de gradiente", "gradient descent", "un método que ajusta los parámetros en la dirección que reduce la pérdida", "a method that adjusts parameters in the direction that reduces the loss"),
    ("el sobreajuste", "overfitting", "el fenómeno por el que un modelo memoriza el entrenamiento y generaliza mal", "the phenomenon where a model memorizes the training set and generalizes poorly"),
    ("la validación cruzada", "cross-validation", "una técnica que estima el rendimiento repartiendo los datos en varios pliegues", "a technique that estimates performance by splitting the data into several folds"),
    ("el tokenizador", "the tokenizer", "el componente que convierte el texto en una secuencia de identificadores enteros", "the component that turns text into a sequence of integer identifiers"),
    ("la ventana de contexto", "the context window", "la cantidad máxima de tokens que un modelo puede leer a la vez", "the maximum number of tokens a model can read at once"),
    ("el compilador", "the compiler", "un programa que traduce código fuente a instrucciones de máquina", "a program that translates source code into machine instructions"),
    ("el sistema de archivos", "the file system", "la capa que organiza cómo se guardan y se nombran los datos en disco", "the layer that organizes how data is stored and named on disk"),
    ("la memoria virtual", "virtual memory", "una abstracción que da a cada proceso su propio espacio de direcciones", "an abstraction that gives each process its own address space"),
    ("el protocolo TCP", "the TCP protocol", "un protocolo que entrega bytes de forma ordenada y fiable entre dos máquinas", "a protocol that delivers bytes reliably and in order between two machines"),
    ("el cifrado simétrico", "symmetric encryption", "un esquema que usa la misma clave para cifrar y descifrar", "a scheme that uses the same key to encrypt and decrypt"),
    ("las pruebas unitarias", "unit tests", "comprobaciones automáticas que verifican una función aislada", "automatic checks that verify a single isolated function"),
    ("la integración continua", "continuous integration", "una práctica que compila y prueba cada cambio de forma automática", "a practice that builds and tests every change automatically"),
    ("los microservicios", "microservices", "una arquitectura que divide la aplicación en servicios pequeños e independientes", "an architecture that splits the application into small independent services"),
    ("la paginación", "pagination", "una técnica que devuelve los resultados en bloques de tamaño fijo", "a technique that returns results in fixed-size blocks"),
    ("la limitación de tasa", "rate limiting", "una política que restringe cuántas peticiones puede hacer un cliente por minuto", "a policy that restricts how many requests a client can make per minute"),
    ("la serialización JSON", "JSON serialization", "el proceso de convertir objetos en texto JSON y viceversa", "the process of converting objects to JSON text and back"),
    ("la variable de entorno", "the environment variable", "un par nombre-valor que el sistema operativo entrega a cada proceso", "a name-value pair that the operating system hands to each process"),
    ("el entorno virtual", "the virtual environment", "una carpeta aislada con su propio intérprete de Python y sus paquetes", "an isolated folder with its own Python interpreter and packages"),
    ("el decorador", "the decorator", "una función que envuelve a otra para añadirle comportamiento", "a function that wraps another one to add behavior"),
    ("el generador", "the generator", "una función que produce valores perezosamente con la palabra yield", "a function that lazily produces values with the yield keyword"),
    ("el manejo de excepciones", "exception handling", "el mecanismo para capturar errores sin que el programa se detenga", "the mechanism for catching errors without stopping the program"),
    ("la programación asíncrona", "asynchronous programming", "un modelo que solapa esperas de entrada y salida sin bloquear el hilo", "a model that overlaps input and output waits without blocking the thread"),
    ("el bloqueo de base de datos", "the database lock", "un candado que impide que dos transacciones modifiquen la misma fila a la vez", "a lock that prevents two transactions from modifying the same row at once"),
    ("la migración de esquema", "the schema migration", "un script versionado que cambia la estructura de la base de datos", "a versioned script that changes the structure of the database"),
    ("el registro de eventos", "event logging", "la práctica de escribir mensajes con marca de tiempo sobre lo que hace el sistema", "the practice of writing timestamped messages about what the system does"),
]
HERRAMIENTAS = ["PostgreSQL", "Redis", "nginx", "Docker", "pytest", "Kubernetes", "FastAPI", "SQLite", "Git", "Linux", "Celery", "Prometheus"]
PARAMETROS = ["max_connections", "timeout_seconds", "cache_size", "batch_size", "log_level", "retry_count", "pool_size", "num_workers"]
# aspecto -> (título ES, título EN, palabra clave ES, palabra clave EN, plantillas ES, plantillas EN)
ASPECTOS = {
    "intro": ("{n}: qué es y para qué sirve", "{n}: what it is and what it is for", "qué es", "what is",
        ["Se usa en {t} para simplificar el trabajo diario del equipo.", "Entender {c} es un requisito antes de pasar a producción.", "Su idea central es sencilla y aparece en casi todos los sistemas modernos.", "La documentación de {t} dedica un capítulo completo a {c}."],
        ["It is used in {t} to simplify the daily work of the team.", "Understanding {c} is a prerequisite before going to production.", "Its core idea is simple and appears in almost every modern system.", "The {t} documentation devotes a whole chapter to {c}."]),
    "config": ("{n}: cómo configurarlo", "{n}: how to configure it", "configurar", "configure",
        ["Para configurar {c} edita el archivo de ajustes y reinicia {t}.", "El parámetro {p} controla el comportamiento de {c} y su valor por defecto es {x}.", "Después de cambiar {p} a {x}, comprueba el resultado con un comando de diagnóstico.", "Guarda la configuración de {c} en un archivo versionado y no en la consola."],
        ["To configure {c}, edit the settings file and restart {t}.", "The {p} parameter controls the behavior of {c} and its default value is {x}.", "After changing {p} to {x}, check the result with a diagnostic command.", "Keep the configuration of {c} in a versioned file and not in the console."]),
    "errores": ("{n}: errores frecuentes", "{n}: common errors", "errores", "errors",
        ["Un error frecuente con {c} es olvidar el valor de {p} y dejarlo en {x}.", "Si aparece un fallo intermitente en {t}, revisa primero los registros de {c}.", "Muchos equipos descubren el problema de {c} solo cuando la carga se multiplica por {x}.", "El mensaje de error suele ser engañoso: la causa real está en {c}, no en {t}."],
        ["A common mistake with {c} is forgetting {p} and leaving it at {x}.", "If an intermittent failure shows up in {t}, check the logs of {c} first.", "Many teams only discover the problem with {c} when the load grows by {x} times.", "The error message is usually misleading: the real cause is in {c}, not in {t}."]),
    "rendimiento": ("{n}: rendimiento y optimización", "{n}: performance and optimization", "rendimiento", "performance",
        ["Para medir el rendimiento de {c} usa un perfilador y no tu intuición.", "Subir {p} a {x} mejora la latencia, pero consume más memoria en {t}.", "La optimización prematura de {c} suele complicar el código sin ganar tiempo.", "En una prueba con {x} usuarios, {c} redujo la latencia mediana a la mitad."],
        ["To measure the performance of {c}, use a profiler and not your intuition.", "Raising {p} to {x} improves latency but uses more memory in {t}.", "Premature optimization of {c} usually complicates the code without saving time.", "In a test with {x} users, {c} cut the median latency in half."]),
    "practicas": ("{n}: buenas prácticas", "{n}: best practices", "buenas prácticas", "best practices",
        ["Una buena práctica con {c} es documentar {p} y automatizar su despliegue en {t}.", "Revisa {c} en cada revisión de código y escribe pruebas para los casos límite.", "Mantén {c} simple: añade complejidad solo cuando los datos lo justifiquen.", "Define alertas sobre {c} antes del primer incidente, no después."],
        ["A good practice with {c} is to document {p} and automate its deployment in {t}.", "Review {c} in every code review and write tests for the edge cases.", "Keep {c} simple: add complexity only when the data justifies it.", "Define alerts on {c} before the first incident, not after."]),
    "ejemplo": ("{n}: ejemplo en Python", "{n}: Python example", "ejemplo", "example",
        ["En este ejemplo en Python usamos {t} para mostrar {c} paso a paso.", "El script define {p} = {x} y luego imprime el resultado de {c}.", "Ejecuta el ejemplo con Python 3.11 y compara la salida con la esperada.", "Puedes adaptar el código de {c} a tu propio proyecto cambiando solo {p}."],
        ["In this Python example we use {t} to show {c} step by step.", "The script defines {p} = {x} and then prints the result of {c}.", "Run the example with Python 3.11 and compare the output with the expected one.", "You can adapt the code for {c} to your own project by changing only {p}."]),
    "seguridad": ("{n}: consideraciones de seguridad", "{n}: security considerations", "seguridad", "security",
        ["Desde el punto de vista de la seguridad, {c} nunca debe exponerse sin autenticación.", "Rota las credenciales de {t} cada {x} días y no las guardes en el repositorio.", "Un ataque contra {c} suele aprovechar el valor de {p} mal configurado.", "Audita los accesos a {c} y conserva los registros durante {x} días."],
        ["From a security standpoint, {c} must never be exposed without authentication.", "Rotate the {t} credentials every {x} days and do not store them in the repository.", "An attack on {c} usually exploits a misconfigured {p} value.", "Audit access to {c} and keep the logs for {x} days."]),
}

In [ ]:
# ── Generador del corpus sintético (mismo esquema que el real) ────────────────
import random


def _es_fix(s: str) -> str:
    """Contracciones del español: 'a el' -> 'al', 'de el' -> 'del'."""
    return s.replace(" a el ", " al ").replace(" de el ", " del ").replace("De el ", "Del ").replace("A el ", "Al ")


def _sintetico_wiki(lang: str, n_docs: int, n_queries: int, seed: int = 42):
    """Docs técnicos sintéticos ES/EN: concepto x aspecto x herramienta. Devuelve (docs, queries, qrels)."""
    rng = random.Random(seed + (0 if lang == "es" else 1))
    li = 0 if lang == "es" else 1
    combos = [(c, a, t) for c in range(len(CONCEPTOS)) for a in ASPECTOS for t in range(len(HERRAMIENTAS))]
    rng.shuffle(combos)
    docs, titles = [], []
    for k, (ci, asp, ti) in enumerate(combos[:n_docs]):
        concepto, definicion = CONCEPTOS[ci][li], CONCEPTOS[ci][2 + li]
        tool = HERRAMIENTAS[ti]
        tit_t, plantillas = ASPECTOS[asp][li], ASPECTOS[asp][4 + li]
        nombre = concepto.split(" ", 1)[1] if concepto.split(" ", 1)[0] in ("el", "la", "los", "las", "the") else concepto
        slots = lambda: dict(c=concepto, t=tool, p=rng.choice(PARAMETROS), x=rng.choice([8, 16, 30, 64, 100, 256, 1024]))
        frases = [("{C} " + ("es " if li == 0 else "is ") + definicion + ".").format(C=concepto)]
        frases += [pl.format(**slots()) for pl in rng.sample(plantillas, len(plantillas))]
        frases.append(rng.choice(ASPECTOS["intro"][4 + li]).format(**slots()))
        for _ in range(2):  # ruido: frases de otro concepto (hace el retrieval no trivial)
            otro = rng.randrange(len(CONCEPTOS))
            otra = rng.choice(list(ASPECTOS.values()))[4 + li]
            s = slots(); s["c"] = CONCEPTOS[otro][li]
            frases.append(rng.choice(otra).format(**s))
        s = slots()
        frases.append(("Ejemplo: `{p} = {x}`." if li == 0 else "Example: `{p} = {x}`.").format(**s))
        text = " ".join(f[0].upper() + f[1:] for f in frases)
        docs.append({"id": f"{lang}-syn-{k:04d}", "lang": lang, "text": _es_fix(text) if lang == "es" else text})
        titles.append((nombre, ASPECTOS[asp][2 + li], tool, tit_t.format(n=nombre) + f" ({tool})"))
    queries, qrels = [], {}
    for j, k in enumerate(rng.sample(range(len(docs)), min(n_queries, len(docs)))):
        nombre, kw, tool, titulo = titles[k]
        q = titulo.replace(":", "").replace("(", "").replace(")", "") if j % 2 == 0 else f"{nombre} {kw} {tool}"
        qid = f"{lang}-q{j:04d}"
        queries.append({"qid": qid, "lang": lang, "text": q})
        qrels[qid] = {docs[k]["id"]: 1}
    return docs, queries, qrels

In [ ]:
# ── carga_corpus(): cadena oficial (HF) -> espejo (API de Wikipedia / stdlib) -> sintético ─────
import ast
import json
import os
import random
import urllib.parse
import urllib.request

SEED = 42


def _strip_docstring(func_src: str):
    """Devuelve (código sin docstring, primera frase del docstring) o None si no hay docstring útil."""
    try:
        fn = ast.parse(func_src).body[0]
        doc = ast.get_docstring(fn)
        first = fn.body[0]
    except Exception:
        return None
    if not doc or not (isinstance(first, ast.Expr) and isinstance(getattr(first, "value", None), ast.Constant)):
        return None
    lines = func_src.splitlines()
    code = "\n".join(lines[: first.lineno - 1] + lines[first.end_lineno :])
    frase = doc.strip().split("\n\n")[0].replace("\n", " ").strip()
    frase = frase.split(". ")[0].rstrip(".")
    return (code, frase) if len(frase.split()) >= 4 else None


def _code_stdlib(n_docs: int, n_queries: int, rng: random.Random):
    """Espejo OFFLINE con código real: funciones de la biblioteca estándar con su docstring como query."""
    import importlib, inspect, textwrap
    mods = ["argparse", "bisect", "calendar", "collections", "colorsys", "copy", "csv", "difflib", "fnmatch", "fractions",
            "functools", "getpass", "glob", "heapq", "html", "json", "operator", "os", "pathlib", "posixpath", "queue",
            "random", "re", "shlex", "shutil", "statistics", "string", "tempfile", "textwrap", "urllib.parse", "uuid", "zipfile"]
    cands = []
    for name in mods:
        try:
            src = inspect.getsource(importlib.import_module(name))
            tree = ast.parse(src)
        except Exception:
            continue
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.end_lineno - node.lineno >= 4:
                seg = textwrap.dedent("\n".join(src.splitlines()[node.lineno - 1 : node.end_lineno]))
                r = _strip_docstring(seg)
                if r and 200 <= len(r[0]) <= 2500:
                    cands.append((f"{name}.{node.name}", r[0], r[1]))
    rng.shuffle(cands)
    return cands[:n_docs]


def _code_hf(n_docs: int, rng: random.Random):
    """Oficial: CodeSearchNet (Python) en streaming. Prueba varios repos por cambios de formato en `datasets`."""
    from datasets import load_dataset
    errs = []
    for repo, kw in [("code-search-net/code_search_net", {"name": "python"}), ("code_search_net", {"name": "python"}),
                     ("claudios/code_search_net", {"name": "python"}), ("Nan-Do/code-search-net-python", {})]:
        try:
            ds = load_dataset(repo, split="train", streaming=True, **kw).shuffle(seed=SEED, buffer_size=2000)
            out = []
            for ex in ds:
                src = ex.get("func_code_string") or ex.get("whole_func_string") or ex.get("code") or ""
                r = _strip_docstring(src) if '"""' in src or "'''" in src else None
                doc = (ex.get("func_documentation_string") or ex.get("docstring") or "").strip().split("\n\n")[0]
                doc = doc.replace("\n", " ").split(". ")[0].rstrip(".")
                code = r[0] if r else src
                if 200 <= len(code) <= 2500 and len(doc.split()) >= 4:
                    out.append((ex.get("func_name") or f"csn.{len(out)}", code, doc))
                if len(out) >= n_docs:
                    return out
        except Exception as e:  # noqa: BLE001
            errs.append(f"{repo}: {type(e).__name__}")
    raise RuntimeError("CodeSearchNet no disponible -> " + "; ".join(errs))


def _wiki_hf(lang: str, n_docs: int, cap: int):
    """Oficial: wikimedia/wikipedia (snapshot 20231101) en streaming con un buffer pequeño."""
    from datasets import load_dataset
    ds = load_dataset("wikimedia/wikipedia", f"20231101.{lang}", split="train", streaming=True)
    ds = ds.shuffle(seed=SEED, buffer_size=2000)
    out = []
    for ex in ds:
        if len(ex["text"]) >= 600 and len(ex["title"]) >= 3:
            out.append((f"{lang}-{ex['id']}", ex["title"], ex["text"][:cap]))
        if len(out) >= n_docs:
            break
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out


def _wiki_api(lang: str, n_docs: int):
    """Espejo: API pública de Wikipedia (artículos aleatorios, solo la sección introductoria; ~20 por petición)."""
    out, seen, tries = [], set(), 0
    while len(out) < n_docs and tries < 80:
        tries += 1
        qs = urllib.parse.urlencode({"action": "query", "format": "json", "generator": "random", "grnnamespace": 0,
                                     "grnlimit": 20, "prop": "extracts", "exintro": 1, "explaintext": 1, "exlimit": 20})
        req = urllib.request.Request(f"https://{lang}.wikipedia.org/w/api.php?{qs}",
                                     headers={"User-Agent": "poliglota-course/1.0 (curso educativo)"})
        with urllib.request.urlopen(req, timeout=20) as r:
            pages = json.load(r).get("query", {}).get("pages", {})
        for p in pages.values():
            if p["pageid"] not in seen and len(p.get("extract", "")) >= 300:
                seen.add(p["pageid"])
                out.append((f"{lang}-{p['pageid']}", p["title"], p["extract"]))
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out[:n_docs]


def carga_corpus(fast_dev: bool = True) -> dict:
    """Corpus Políglota -> {"docs": [{id, lang, text}], "queries": [{qid, lang, text}], "qrels": {qid: {doc_id: 1}},
    "source": "real" | "synthetic"}. Cadena: oficial (HF) -> espejo (API Wikipedia; código de la stdlib) -> sintético."""
    rng = random.Random(SEED)
    n_wiki, n_code, q_wiki, q_code, cap = (150, 100, 40, 30, 1500) if fast_dev else (3000, 1500, 200, 150, 6000)
    docs, queries, qrels, source = [], [], {}, "real"
    wiki = {}
    for lang in ("es", "en"):
        for nombre, fn in (("HF wikimedia/wikipedia", lambda: _wiki_hf(lang, n_wiki, cap)),
                           ("API de Wikipedia (espejo)", lambda: _wiki_api(lang, min(n_wiki, 400)))):
            try:
                wiki[lang] = fn()
                print(f"[carga_corpus] Wikipedia {lang}: {len(wiki[lang])} artículos desde {nombre}")
                break
            except Exception as e:  # noqa: BLE001
                print(f"[carga_corpus] {nombre} ({lang}) falló: {type(e).__name__}: {str(e)[:80]}")
    if len(wiki) == 2:
        for lang, arts in wiki.items():
            docs += [{"id": i, "lang": lang, "text": t} for i, _, t in arts]
            for j, k in enumerate(rng.sample(range(len(arts)), min(q_wiki, len(arts)))):
                qid = f"{lang}-q{j:04d}"
                queries.append({"qid": qid, "lang": lang, "text": arts[k][1]})
                qrels[qid] = {arts[k][0]: 1}
    else:
        source = "synthetic"
        print("⚠️  [carga_corpus] SIN acceso a Wikipedia: uso texto ES/EN SINTÉTICO (mismo esquema). Los números NO son comparables con datos reales.")
        for lang in ("es", "en"):
            d, q, r = _sintetico_wiki(lang, n_wiki, q_wiki, SEED)
            docs += d; queries += q; qrels.update(r)
    try:
        code = _code_hf(n_code, rng)
        print(f"[carga_corpus] código: {len(code)} funciones desde CodeSearchNet (HF)")
    except Exception as e:  # noqa: BLE001
        print(f"[carga_corpus] CodeSearchNet falló ({str(e)[:90]}); uso funciones REALES de la stdlib de Python (espejo offline)")
        code = _code_stdlib(n_code, q_code, rng)
    for k, (name, src, doc) in enumerate(code):
        docs.append({"id": f"code-{k:04d}", "lang": "code", "text": src})
    for j, k in enumerate(rng.sample(range(len(code)), min(q_code, len(code)))):
        qid = f"code-q{j:04d}"
        queries.append({"qid": qid, "lang": "code", "text": code[k][2]})
        qrels[qid] = {f"code-{k:04d}": 1}
    return {"docs": docs, "queries": queries, "qrels": qrels, "source": source}

In [ ]:
# ── Métricas de retrieval canónicas de Políglota (módulo 00) ──────────────────
import math

import numpy as np


def _dedup(ranked_ids):
    """Quita duplicados conservando la primera aparición (un doc repetido NO puntúa dos veces)."""
    seen, out = set(), []
    for d in ranked_ids:
        if d not in seen:
            seen.add(d)
            out.append(d)
    return out


def dcg_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """DCG con ganancia lineal (como trec_eval / pytrec_eval, que usa BEIR/MTEB) y descuento 1/log2(rango+1)."""
    ranked = _dedup(ranked_ids)[:k]
    return sum(rels.get(d, 0) / math.log2(i + 2) for i, d in enumerate(ranked) if rels.get(d, 0) > 0)


def ndcg_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """nDCG@k. rels: doc_id -> relevancia (>0 relevante). Sin relevantes -> 0.0 (evaluate() las excluye)."""
    ideal = sorted((r for r in rels.values() if r > 0), reverse=True)[:k]
    idcg = sum(r / math.log2(i + 2) for i, r in enumerate(ideal))
    return 0.0 if idcg == 0 else dcg_at_k(ranked_ids, rels, k) / idcg


def recall_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """Fracción de los documentos relevantes que aparecen en el top-k."""
    relevant = {d for d, r in rels.items() if r > 0}
    if not relevant:
        return 0.0
    return len(relevant & set(_dedup(ranked_ids)[:k])) / len(relevant)


def mrr_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """1/rango del primer relevante dentro del top-k (0 si no hay ninguno)."""
    for i, d in enumerate(_dedup(ranked_ids)[:k]):
        if rels.get(d, 0) > 0:
            return 1.0 / (i + 1)
    return 0.0


def rank_from_scores(scores, doc_ids, k: int | None = None):
    """Ordena por score desc. Empates -> orden original del corpus (determinista). Los empates son una fuente
    silenciosa de varianza: con scores iguales, el orden de entrada decidiría el resultado."""
    scores = np.asarray(scores, dtype=float)
    order = np.lexsort((np.arange(len(scores)), -scores))
    order = order[:k] if k else order
    return [doc_ids[i] for i in order]


def evaluate(run: dict, qrels: dict, k: int = 10) -> dict:
    """run: {qid: [doc_id ordenados]}. Promedia solo sobre queries con algún relevante (convención trec_eval)."""
    n, tot = 0, {"ndcg": 0.0, "recall": 0.0, "mrr": 0.0}
    for qid, rels in qrels.items():
        if not any(r > 0 for r in rels.values()):
            continue
        ranked = run.get(qid, [])
        tot["ndcg"] += ndcg_at_k(ranked, rels, k)
        tot["recall"] += recall_at_k(ranked, rels, k)
        tot["mrr"] += mrr_at_k(ranked, rels, k)
        n += 1
    out = {m: (v / n if n else 0.0) for m, v in tot.items()}
    out.update(n_queries=n, n_skipped=len(qrels) - n)
    return out

In [ ]:
# ── Tokenización simple + BM25 propio (módulo 00) ─────────────────────────────
import math
import re
import unicodedata
from collections import Counter, defaultdict

import numpy as np

_WORD_RE = re.compile(r"\w+", re.UNICODE)


def fold_accents(text: str) -> str:
    """'canción' -> 'cancion' (NFD y descartar marcas combinantes Mn)."""
    nfd = unicodedata.normalize("NFD", text)
    return unicodedata.normalize("NFC", "".join(c for c in nfd if unicodedata.category(c) != "Mn"))


def tokenize_simple(text: str, accents: str = "keep") -> list[str]:
    """Minúsculas + palabras \\w+. accents='fold' quita tildes. (Esto NO es el 'token' de un LLM: es un token léxico.)"""
    text = unicodedata.normalize("NFC", text).lower()
    if accents == "fold":
        text = fold_accents(text)
    return _WORD_RE.findall(text)


class BM25:
    """BM25 con índice invertido. idf='lucene': ln(1+(N-n+.5)/(n+.5)) (siempre >= 0).
    idf='okapi': ln((N-n+.5)/(n+.5)) con suelo epsilon*idf_medio (como rank_bm25.BM25Okapi)."""

    def __init__(self, k1: float = 1.2, b: float = 0.75, idf: str = "lucene", epsilon: float = 0.25):
        self.k1, self.b, self.idf_kind, self.epsilon = k1, b, idf, epsilon

    def fit(self, docs_tokens: list[list[str]], doc_ids: list[str]) -> "BM25":
        self.doc_ids = list(doc_ids)
        self.N = len(docs_tokens)
        self.dl = np.array([len(t) for t in docs_tokens], dtype=float)
        self.avgdl = float(self.dl.mean()) if self.N else 0.0
        self.postings = defaultdict(list)  # término -> [(índice_doc, tf)]
        for i, toks in enumerate(docs_tokens):
            for term, tf in Counter(toks).items():
                self.postings[term].append((i, tf))
        raw = {t: len(p) for t, p in self.postings.items()}
        if self.idf_kind == "lucene":
            self.idf = {t: math.log(1 + (self.N - n + 0.5) / (n + 0.5)) for t, n in raw.items()}
        else:
            self.idf = {t: math.log(self.N - n + 0.5) - math.log(n + 0.5) for t, n in raw.items()}
            floor = self.epsilon * sum(self.idf.values()) / len(self.idf)
            self.idf = {t: (v if v >= 0 else floor) for t, v in self.idf.items()}
        return self

    def scores(self, q_tokens: list[str]) -> np.ndarray:
        s = np.zeros(self.N)
        for term in q_tokens:  # los términos repetidos de la query suman varias veces (igual que rank_bm25)
            for i, tf in self.postings.get(term, ()):
                norm = self.k1 * (1 - self.b + self.b * self.dl[i] / self.avgdl)
                s[i] += self.idf[term] * tf * (self.k1 + 1) / (tf + norm)
        return s

    def search(self, q_tokens: list[str], k: int = 10) -> list[str]:
        s = self.scores(q_tokens)
        order = np.lexsort((np.arange(self.N), -s))[:k]  # empates -> orden del corpus
        return [self.doc_ids[i] for i in order]

In [ ]:
def bootstrap_ci(values, n_boot: int = 2000, alpha: float = 0.05, seed: int = 42):
    """IC percentil de la media remuestreando QUERIES (la unidad de muestreo es la query, no el par query-doc)."""
    v = np.asarray(values, dtype=float)
    idx = np.random.default_rng(seed).integers(0, len(v), size=(n_boot, len(v)))
    means = v[idx].mean(axis=1)
    return float(v.mean()), float(np.quantile(means, alpha / 2)), float(np.quantile(means, 1 - alpha / 2))

def paired_bootstrap(a, b, n_boot: int = 10000, alpha: float = 0.05, seed: int = 42) -> dict:
    """Test pareado A vs B sobre las MISMAS queries. IC percentil de la diferencia media y p-valor bilateral
    por 'shift' (se centra la diferencia en 0 bajo H0 y se cuenta cuántas veces el remuestreo supera lo observado)."""
    d = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(d), size=(n_boot, len(d)))
    means = d[idx].mean(axis=1)
    null = (d - d.mean())[idx].mean(axis=1)
    p = (np.sum(np.abs(null) >= abs(d.mean())) + 1) / (n_boot + 1)
    return {"diff": float(d.mean()), "ci_low": float(np.quantile(means, alpha / 2)),
            "ci_high": float(np.quantile(means, 1 - alpha / 2)), "p_value": float(p)}

def paired_permutation(a, b, n_perm: int = 10000, seed: int = 42) -> float:
    """Test de aleatorización pareado (sign-flip): bajo H0 el signo de cada diferencia por query es intercambiable."""
    d = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    signs = np.random.default_rng(seed).choice([-1.0, 1.0], size=(n_perm, len(d)))
    stat = np.abs((signs * d).mean(axis=1))
    return float((np.sum(stat >= abs(d.mean()) - 1e-12) + 1) / (n_perm + 1))

def holm(pvalues: dict, alpha: float = 0.05) -> dict:
    """Corrección de Holm-Bonferroni para comparaciones múltiples. Devuelve {nombre: (p_ajustado, rechaza_H0)}."""
    items = sorted(pvalues.items(), key=lambda kv: kv[1])
    m, out, running = len(items), {}, 0.0
    for i, (name, p) in enumerate(items):
        running = max(running, min(1.0, (m - i) * p))
        out[name] = (running, running < alpha)
    return out

In [ ]:
# Catálogo de embedders multilingües "pequeños" (caben en CPU/T4). Prefijos copiados de las tarjetas de modelo:
# VERIFÍCALOS en la tarjeta antes de usar un modelo en producción (cambian entre versiones).
MODELOS = {
    "multilingual-e5-small": dict(hf="intfloat/multilingual-e5-small", q="query: ", d="passage: "),
    "multilingual-e5-base": dict(hf="intfloat/multilingual-e5-base", q="query: ", d="passage: "),
    "paraphrase-multilingual-MiniLM-L12-v2": dict(hf="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", q="", d=""),
    "gte-multilingual-base": dict(hf="Alibaba-NLP/gte-multilingual-base", q="", d="", trust=True),
    "bge-m3": dict(hf="BAAI/bge-m3", q="", d=""),
    "nomic-embed-text-v2-moe": dict(hf="nomic-ai/nomic-embed-text-v2-moe", q="search_query: ", d="search_document: ", trust=True),
    "Qwen3-Embedding-0.6B": dict(hf="Qwen/Qwen3-Embedding-0.6B", d="",
                                 q="Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:"),
    "embeddinggemma-300m": dict(hf="google/embeddinggemma-300m", q="task: search result | query: ", d="title: none | text: ", gated=True),
}

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

class HashSVDEmbedder:
    """Fallback SIN descargas: TF-IDF de n-gramas de caracteres + SVD. NO es multilingüe ni semántico:
    sirve para ejercitar el pipeline (truncado, cuantización, métricas) cuando HF no es accesible."""
    def __init__(self, dim: int = 64, ngram=(3, 5), analyzer: str = "char_wb"):
        self.dim = dim
        self.tfidf = TfidfVectorizer(analyzer=analyzer, ngram_range=ngram, sublinear_tf=True, min_df=1)
        self.svd = TruncatedSVD(n_components=dim, random_state=SEED)
    def fit(self, textos):
        X = self.tfidf.fit_transform(textos)
        self.svd.n_components = min(self.dim, X.shape[1] - 1, X.shape[0] - 1)
        self.svd.fit(X)
        return self
    def encode(self, textos):
        Z = self.svd.transform(self.tfidf.transform(textos))
        return (Z / np.maximum(np.linalg.norm(Z, axis=1, keepdims=True), 1e-9)).astype(np.float32)

class Embedder:
    """Interfaz común: encode_queries / encode_docs (aplican el prefijo de cada modelo) -> float32 normalizado."""
    def __init__(self, clave, cfg, model=None, fallback=None):
        self.clave, self.cfg, self.model, self.fallback = clave, cfg, model, fallback
        self.is_real = model is not None
    def _enc(self, textos, prefijo, bs=16):
        if self.model is not None:
            E = self.model.encode([prefijo + t for t in textos], batch_size=bs, normalize_embeddings=True,
                                  convert_to_numpy=True, show_progress_bar=False)
            return E.astype(np.float32)
        return self.fallback.encode([prefijo + t for t in textos])
    def encode_queries(self, textos): return self._enc(textos, self.cfg.get("q", ""))
    def encode_docs(self, textos): return self._enc(textos, self.cfg.get("d", ""))
    @property
    def n_params(self):
        return sum(p.numel() for p in self.model.parameters()) if self.model is not None else 0

def cargar_embedder(clave, textos_ajuste=None, max_len=256):
    """Carga con sentence-transformers; si no se puede (sin red, modelo con licencia, falta de RAM) cae al fallback."""
    cfg = MODELOS[clave]
    try:
        from sentence_transformers import SentenceTransformer
        m = SentenceTransformer(cfg["hf"], device=device, trust_remote_code=cfg.get("trust", False))
        m.max_seq_length = min(m.max_seq_length or max_len, max_len)
        return Embedder(clave, cfg, model=m)
    except Exception as e:
        print(f"[embedder] {clave}: no se pudo cargar ({type(e).__name__}: {str(e)[:70]})")
        if textos_ajuste is None:
            return None
        print(f"[embedder] {clave}: usando fallback TF-IDF+SVD (NO representa al modelo real)")
        return Embedder(clave + " (fallback)", cfg, fallback=HashSVDEmbedder().fit(textos_ajuste))

In [ ]:
corpus = carga_corpus(fast_dev=FAST_DEV_RUN)
docs, queries, qrels = corpus["docs"], corpus["queries"], corpus["qrels"]
doc_ids = [d["id"] for d in docs]; doc_texts = [d["text"] for d in docs]
doc_lang = np.array([d["lang"] for d in docs])
qids = [q["qid"] for q in queries]; q_texts = [q["text"] for q in queries]; q_lang = np.array([q["lang"] for q in queries])
print(f"fuente={corpus['source']} · {len(docs)} docs · {len(queries)} queries · idiomas={sorted(set(doc_lang))}")

In [ ]:
def run_desde_scores(S, k: int = 10, ids=None) -> dict:
    """Matriz de scores (n_queries x n_docs) -> run {qid: [doc_ids top-k]} con el desempate canónico."""
    ids = doc_ids if ids is None else ids
    return {qid: rank_from_scores(S[i], ids, k) for i, qid in enumerate(qids)}

def ndcg_por_query(run: dict, k: int = 10) -> np.ndarray:
    """nDCG@k de cada query (orden de `qids`); unidad de remuestreo para bootstrap."""
    return np.array([ndcg_at_k(run.get(q, []), qrels[q], k) for q in qids])

def l2n(X):
    X = np.asarray(X, dtype=np.float32)
    return X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-9)

BM25_IDX = BM25().fit([tokenize_simple(t) for t in doc_texts], doc_ids)
S_BM25 = np.array([BM25_IDX.scores(tokenize_simple(q)) for q in q_texts])
print("BM25 (baseline del módulo 00):", {m: round(v, 3) for m, v in evaluate(run_desde_scores(S_BM25), qrels).items() if m in ("ndcg", "recall", "mrr")})

In [ ]:
# Dos versiones del embedder de Políglota: v1 (la que está en producción) y v2 (la candidata).
# Con hub: v1 = multilingual-e5-small (384 d), v2 = multilingual-e5-base (768 d).
# Sin hub: dos fallbacks TF-IDF+SVD DISTINTOS (otro analizador y otra dimensión) -> espacios incompatibles, como en la vida real.
USAR_MODELOS_REALES = True
V1_REAL, V2_REAL = "multilingual-e5-small", "multilingual-e5-base"

def carga_versiones(textos_ajuste) -> dict:
    vers = {}
    if USAR_MODELOS_REALES:
        for nombre, clave in (("v1", V1_REAL), ("v2", V2_REAL)):
            e = cargar_embedder(clave)          # None si no hay hub
            if e is None:
                break
            vers[nombre] = e
    if len(vers) < 2:
        print("⚠️  Sin modelos reales: v1/v2 = fallbacks TF-IDF+SVD (NO son modelos neuronales; las cifras no representan a e5).")
        vers = {"v1": Embedder("v1 · char 3-4 + SVD 48d (fallback)", {}, fallback=HashSVDEmbedder(dim=48, ngram=(3, 4)).fit(textos_ajuste)),
                "v2": Embedder("v2 · char 2-5 + SVD 128d (fallback)", {}, fallback=HashSVDEmbedder(dim=128, ngram=(2, 5)).fit(textos_ajuste))}
    return vers

VERS = carga_versiones(doc_texts)
EMB = {}
for v, e in VERS.items():
    t0 = time.perf_counter()
    D, Qv = e.encode_docs(doc_texts), e.encode_queries(q_texts)
    EMB[v] = {"D": D, "Q": Qv, "seg": time.perf_counter() - t0}
    print(f"{v}: {e.clave} · dim={D.shape[1]} · nDCG@10={evaluate(run_desde_scores(Qv @ D.T), qrels)['ndcg']:.3f} · {EMB[v]['seg']:.1f}s")

In [ ]:
from collections import OrderedDict

def normaliza_query(texto: str) -> str:
    """Normalización ANTES de la clave de caché: NFC + minúsculas + espacios colapsados (sube el hit rate)."""
    return " ".join(unicodedata.normalize("NFC", texto).lower().split())

def clave_cache(version: str, texto: str) -> str:
    """La versión del modelo VA EN LA CLAVE: si no, tras migrar servirías vectores v1 contra un índice v2."""
    return hashlib.sha1(f"{version}\x00{normaliza_query(texto)}".encode("utf-8")).hexdigest()

class LRUCache:
    """Caché LRU O(1) con OrderedDict y contadores de aciertos/fallos."""
    def __init__(self, capacidad: int):
        self.cap, self.d, self.hits, self.misses = capacidad, OrderedDict(), 0, 0
    def get(self, k):
        if k in self.d:
            self.d.move_to_end(k); self.hits += 1
            return self.d[k]
        self.misses += 1
        return None
    def put(self, k, v):
        self.d[k] = v; self.d.move_to_end(k)
        if len(self.d) > self.cap:
            self.d.popitem(last=False)          # expulsa el menos usado recientemente
    @property
    def hit_rate(self) -> float:
        n = self.hits + self.misses
        return self.hits / n if n else 0.0

In [ ]:
import faiss
faiss.omp_set_num_threads(1)

class ServicioPoliglota:
    """Búsqueda versionada: caché de embeddings de query + índice denso por versión (FAISS IndexFlatIP) + BM25,
    filtro por idioma, fusión RRF, alias 'prod' -> versión y versiones 'adaptador' (queries nuevas -> índice viejo)."""
    def __init__(self, docs, bm25, cache_cap: int = 2048, rrf_k: int = 60, profundidad: int = 50, tokenizador=None,
                 fusion: str = "rrf", alfa: float = 0.5):
        # CONTRATO: `tokenizador` debe ser el MISMO con el que se construyó `bm25` (si no, fallo silencioso F3 del módulo 19)
        self.tok = tokenizador or tokenize_simple
        self.fusion, self.alfa = fusion, alfa      # fusion: 'rrf' | 'lineal' (alfa·denso + (1-alfa)·BM25, min-max en cada lista)
        self.ids = [d["id"] for d in docs]; self.lang = np.array([d["lang"] for d in docs])
        self.bm25, self.cache, self.rrf_k, self.prof = bm25, LRUCache(cache_cap), rrf_k, profundidad
        self.versiones, self.alias, self.lat_ms = {}, {}, []

    def registrar_version(self, nombre, embedder, D, indice: str = "flat", ef_search: int = 64):
        """indice: 'flat' (exacto) | 'hnsw' (FAISS IndexHNSWFlat, M=32)."""
        if indice == "hnsw":
            idx = faiss.IndexHNSWFlat(D.shape[1], 32, faiss.METRIC_INNER_PRODUCT); idx.hnsw.efConstruction = 80
        else:
            idx = faiss.IndexFlatIP(D.shape[1])
        idx.add(l2n(D))
        self.versiones[nombre] = {"emb": embedder, "idx": idx, "W": None, "base": nombre, "indice": indice, "ef": ef_search}

    def registrar_adaptador(self, nombre, embedder, W, sobre: str):
        """Queries con `embedder` (versión nueva) proyectadas con W al espacio del índice `sobre` (versión vieja)."""
        base = self.versiones[sobre]
        self.versiones[nombre] = {"emb": embedder, "idx": base["idx"], "W": W, "base": sobre, "indice": base["indice"], "ef": base["ef"]}

    def embebe_query(self, texto, nombre):
        k = clave_cache(nombre, texto)
        v = self.cache.get(k)
        if v is None:
            cfg = self.versiones[nombre]
            v = cfg["emb"].encode_queries([texto])
            if cfg["W"] is not None:
                v = l2n(v @ cfg["W"])
            self.cache.put(k, v)
        return v

    def _rangos_densos(self, texto, nombre, cand):
        return [int(i) for i in self._cands_densos(texto, nombre, cand)[0]]

    def _cands_densos(self, texto, nombre, cand):
        """(índices, scores) del top-`profundidad` denso, con el filtro aplicado DENTRO de la búsqueda."""
        cfg = self.versiones[nombre]; idx = cfg["idx"]; q = self.embebe_query(texto, nombre)
        sel = faiss.IDSelectorBatch(cand.astype("int64")) if cand is not None else None
        if cfg["indice"] == "hnsw":       # HNSW exige sus propios parámetros (efSearch + selector de filtro)
            params = faiss.SearchParametersHNSW(efSearch=cfg["ef"]) if sel is None else faiss.SearchParametersHNSW(sel=sel, efSearch=cfg["ef"])
        else:
            params = faiss.SearchParameters(sel=sel) if sel is not None else None
        Dsc, I = idx.search(q, min(self.prof, idx.ntotal), params=params)
        m = I[0] >= 0
        return I[0][m], Dsc[0][m]

    def _lineal(self, texto, nombre, cand):
        """Combinación convexa con min-max dentro de cada lista (un doc ausente de una lista aporta 0 por ella)."""
        I, sd = self._cands_densos(texto, nombre, cand)
        sb_all = self.bm25.scores(self.tok(texto))
        Ib = np.array(self._rangos_bm25(texto, cand), dtype=int); sb = sb_all[Ib] if len(Ib) else np.zeros(0)
        def mm(v):
            return (v - v.min()) / max(float(v.max() - v.min()), 1e-9) if len(v) else v
        score = {}
        for i, v in zip(I, mm(sd)):
            score[int(i)] = score.get(int(i), 0.0) + self.alfa * float(v)
        for i, v in zip(Ib, mm(sb)):
            score[int(i)] = score.get(int(i), 0.0) + (1 - self.alfa) * float(v)
        return score

    def _rangos_bm25(self, texto, cand):
        s = self.bm25.scores(self.tok(texto))
        if cand is not None:
            m = np.full(len(s), -np.inf); m[cand] = s[cand]; s = m
        orden = np.lexsort((np.arange(len(s)), -s))[: self.prof]
        return [int(i) for i in orden if np.isfinite(s[i]) and s[i] > 0]

    def buscar(self, texto, k: int = 10, lang=None, modo: str = "hibrido", version: str = "prod"):
        t0 = time.perf_counter()
        nombre = self.alias.get(version, version)
        cand = np.flatnonzero(self.lang == lang) if lang else None
        if modo == "hibrido" and self.fusion == "lineal":
            score = self._lineal(texto, nombre, cand)
        else:
            listas = []
            if modo in ("denso", "hibrido"):
                listas.append(self._rangos_densos(texto, nombre, cand))
            if modo in ("bm25", "hibrido"):
                listas.append(self._rangos_bm25(texto, cand))
            score = {}
            for lst in listas:                       # RRF: sum 1/(k + rango), rango desde 1
                for r, i in enumerate(lst):
                    score[i] = score.get(i, 0.0) + 1.0 / (self.rrf_k + r + 1)
        top = sorted(score, key=lambda i: (-score[i], i))[:k]
        self.lat_ms.append((time.perf_counter() - t0) * 1e3)
        return [(self.ids[i], round(score[i], 5)) for i in top]

    def metricas(self) -> dict:
        lat = np.array(self.lat_ms) if self.lat_ms else np.zeros(1)
        return {"n": len(self.lat_ms), "p50_ms": float(np.percentile(lat, 50)), "p99_ms": float(np.percentile(lat, 99)),
                "hit_rate": self.cache.hit_rate, "versiones": sorted(self.versiones), "alias": dict(self.alias)}

In [ ]:
import re
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, normalizers

def split_estratificado(queries, frac_test: float = 0.5, seed: int = SEED):
    """dev / test CIEGO estratificado por idioma (módulo 15). Todo lo que se elige, se elige en dev."""
    rng = random.Random(seed); dev, test = [], []
    for lang in sorted({q["lang"] for q in queries}):
        grp = [q for q in queries if q["lang"] == lang]; rng.shuffle(grp)
        cut = int(round(len(grp) * frac_test)); test += grp[:cut]; dev += grp[cut:]
    return dev, test

DEV, TEST = split_estratificado(queries)
I_DEV = np.array([qids.index(q["qid"]) for q in DEV]); I_TEST = np.array([qids.index(q["qid"]) for q in TEST])
print(f"dev={len(DEV)} · test ciego={len(TEST)} queries (estratificado por idioma)")

def entrena_bpe(textos, vocab=4000):
    """BPE byte-level con HF tokenizers (módulos 02-03): NFC + ByteLevel + BPE."""
    tk = Tokenizer(models.BPE(unk_token=None)); tk.normalizer = normalizers.NFC()
    tk.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True); tk.decoder = decoders.ByteLevel()
    tk.train_from_iterator(textos, trainers.BpeTrainer(vocab_size=vocab, initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), show_progress=False))
    return tk

def tok_ident(texto: str) -> list[str]:
    """Analizador léxico de Políglota: palabras + partes de identificadores (snake_case / camelCase), módulos 00 y 13."""
    out = tokenize_simple(texto)
    for w in re.findall(r"[A-Za-z_][A-Za-z0-9_]*", texto):
        partes = re.findall(r"[A-Z]?[a-z]+|[A-Z]+(?![a-z])|\d+", w.replace("_", " "))
        if len(partes) > 1:
            out += [p.lower() for p in partes]
    return out

class EmbedderRecortado:
    """Toma las primeras `dim` coordenadas y renormaliza (Matryoshka; con el fallback SVD equivale a PCA truncada)."""
    def __init__(self, base, dim):
        self.base, self.dim, self.clave = base, dim, f"{base.clave} [:{dim}]"
    def encode_queries(self, t):
        return l2n(self.base.encode_queries(t)[:, :self.dim])
    def encode_docs(self, t):
        return l2n(self.base.encode_docs(t)[:, :self.dim])

def rrf_mat(*Ss, k=60):
    out = np.zeros_like(Ss[0], dtype=float)
    for S in Ss:
        out += 1.0 / (k + np.argsort(np.argsort(-S, axis=1, kind="stable"), axis=1, kind="stable") + 1)
    return out

def minmax(S):
    lo, hi = S.min(1, keepdims=True), S.max(1, keepdims=True)
    return (S - lo) / np.maximum(hi - lo, 1e-9)

def lineal_lista(S1, S2, a, prof=50):
    """Igual que el servicio: cada lista se corta a su top-`prof`, se normaliza min-max dentro de la lista y los docs
    fuera de una lista aportan 0 por ella. S1 = denso, S2 = BM25 (n_queries x n_docs)."""
    def norm(S):
        thr = -np.sort(-S, axis=1)[:, min(prof, S.shape[1]) - 1][:, None]
        Sc = np.maximum(S, thr)
        return (Sc - thr) / np.maximum(Sc.max(1, keepdims=True) - thr, 1e-9)
    return a * norm(S1) + (1 - a) * norm(S2)

def int8_ida_vuelta(D):
    """Cuantiza a int8 (calibrado por dimensión) y decodifica: simula la pérdida de precisión del índice."""
    lo, hi = np.quantile(D, 0.001, axis=0), np.quantile(D, 0.999, axis=0); esc = np.maximum(hi - lo, 1e-9) / 255
    return (np.clip(np.round((D - lo) / esc) - 128, -128, 127) + 128) * esc + lo

pq_S = lambda S: ndcg_por_query(run_desde_scores(S))      # nDCG@10 por query (orden de qids)

## 1. Criterios de aceptación (se fijan ANTES de construir)
Escribirlos primero evita mover la portería después. Son los de la CTO de Políglota Labs para la beta; los umbrales de latencia son para **CPU, 1 hilo, batch = 1** (en GPU/servidor real se re-calibran).

In [ ]:
ACEPTACION = {
    "calidad_vs_bm25":   "nDCG@10 en test ≥ BM25 del módulo 00 y límite inferior del IC95 de la diferencia pareada > −0.03",
    "peor_slice":        "ningún idioma más de 0.05 por debajo de BM25 en test",
    "recall_ann":        "neighbor recall@10 del índice elegido ≥ 0.95 frente a fuerza bruta",
    "latencia_p99_ms":   "p99 del servicio ≤ 50 ms (CPU, batch=1)",
    "ram_10M_gb":        "RAM del índice para 10 M de chunks ≤ 16 GB por réplica",
    "coste_reembed_usd": "re-embedding de 10 M chunks × 400 tokens ≤ 500 $ (API a 0.02 $/1M, oct-2026)",
    "hit_rate":          "hit rate de la caché con tráfico Zipf ≥ 0.70",
}
DECISIONES, METRICAS = {}, {}          # se rellenan sección a sección y alimentan el informe
pd.DataFrame(ACEPTACION.items(), columns=["criterio", "umbral"])

✍️ **Antes de construir, a mano: ¿qué dejan libre los criterios?** Dos de ellos se pueden comprobar con papel:
* **Coste:** $10^7 \cdot 400 = 4\times10^9$ tokens × 0.02 $/1M = **80 $** ≤ 500 $: el re-embedding no restringe nada (como vimos en el módulo 18, lo caro es operar, no embeber). Con una prima de tokens de 1.3 en la mitad española subiría a ≈ 92 $: sigue holgado.
* **RAM ≤ 16 GB para 10 M:** con HNSW (≈ 256 B/vector de grafo) quedan $16\times10^9/10^7 - 256 = 1\,344$ B/vector. En int8 (1 B/dim) cabe hasta $d \approx 1\,344$; en float32 (4 B/dim), solo $d \le 336$. Este criterio es el que de verdad empuja hacia **dimensión recortada + int8** (§3).

Esta es la forma de leer unos criterios de aceptación: identificar cuáles **restringen** el diseño antes de medir nada.

## 2. Tokenizer: coste por idioma y ¿sirve como analizador de BM25?
El tokenizer de Políglota cumple dos papeles distintos que no conviene confundir:
1. **Coste y contexto**: cuántos tokens paga cada idioma (fertilidad, bytes/token) → presupuesto de embedding y de contexto en RAG.
2. **Analizador léxico** para BM25: aquí no se busca comprimir, sino que los términos de la query coincidan con los del documento.

Entrenamos BPE byte-level con HF `tokenizers` (módulos 02–03) con varios tamaños de vocabulario sobre los documentos.

In [ ]:
VOCABS = [1000, 2000, 4000, 8000]
TK = {V: entrena_bpe(doc_texts, V) for V in VOCABS}
filas = []
for V, tk in TK.items():
    for l in ("es", "en", "code"):
        tx = [d["text"] for d in docs if d["lang"] == l]
        n_tok = sum(len(tk.encode(t).ids) for t in tx)
        filas.append({"vocab": V, "lang": l, "fertilidad": n_tok / sum(len(t.split()) for t in tx),
                      "bytes_por_token": sum(len(t.encode()) for t in tx) / n_tok})
df_tok = pd.DataFrame(filas)
fig, axs = plt.subplots(1, 2, figsize=(11, 3.8))
for l, c in zip(("es", "en", "code"), (AZUL, NARANJA, VERDE)):
    d = df_tok[df_tok.lang == l]
    axs[0].plot(d.vocab, d.fertilidad, "o-", color=c, label=l); axs[1].plot(d.vocab, d.bytes_por_token, "o-", color=c, label=l)
for ax, t in zip(axs, ("fertilidad (tokens/palabra)", "bytes por token")):
    ax.set_xscale("log", base=2); ax.set_xlabel("tamaño de vocabulario"); ax.set_ylabel(t); ax.legend()
axs[0].set_title("Fertilidad por idioma"); axs[1].set_title("Compresión por idioma"); plt.tight_layout(); plt.show()
df_tok.pivot(index="vocab", columns="lang", values="fertilidad").round(2)

> 👀 **Qué debes observar:** el código paga muchos más tokens por «palabra» que ES/EN (identificadores, símbolos, indentación) y la brecha se cierra despacio al crecer el vocabulario; ES y EN quedan cerca porque el corpus está equilibrado (en tokenizers entrenados mayoritariamente en inglés, el español paga más, módulo 04). Esta tabla es la que convierte tokens en $ en el informe.

Ahora el segundo papel: comparamos tres analizadores para BM25 **en dev**.

In [ ]:
TK_SEL = TK[4000]
analizadores = {"palabras (módulo 00)": tokenize_simple, "palabras + identificadores": tok_ident,
                "BPE 4k (tokens del LLM)": lambda s: [t.lower() for t in TK_SEL.encode(s).tokens]}
S_LEX, filas = {}, []
for nombre, f in analizadores.items():
    bm = BM25().fit([f(t) for t in doc_texts], doc_ids)
    S_LEX[nombre] = (bm, f, np.array([bm.scores(f(q)) for q in q_texts]))
    pq = pq_S(S_LEX[nombre][2])[I_DEV]
    filas.append({"analizador": nombre, "dev nDCG@10": pq.mean(), **{l: pq[q_lang[I_DEV] == l].mean() for l in ("es", "en", "code")}})
df_lex = pd.DataFrame(filas)
mejor_lex = df_lex.sort_values("dev nDCG@10", ascending=False).iloc[0]["analizador"]
DECISIONES["analizador_bm25"] = mejor_lex
DECISIONES["tokenizer_costo"] = "BPE byte-level 4k (HF tokenizers), solo para presupuesto de tokens"
print("elegido en dev:", mejor_lex)
df_lex.round(3)

> 👀 **Qué debes observar:** los tokens de un BPE de LLM **no** son un buen analizador léxico: trocean palabras en piezas que coinciden por azar entre documentos no relacionados, y BM25 pierde. Partir identificadores (`split_ident`) es lo que ayuda en código. Decisión de ingeniería: el tokenizer del LLM se usa para **costes**, y BM25 usa su propio analizador. (El embedder trae su propio tokenizer: no se cambia sin *tokenizer transfer*, módulo 07.)

## 3. Embedder: modelo, dimensión y precisión (decididos en dev)
Regla de decisión escrita antes de mirar: **(1)** modelo con mejor nDCG@10 en dev si el IC pareado excluye el 0 (si no, el más barato); **(2)** la **menor** dimensión cuya pérdida frente a la completa sea ≤ 0.01; **(3)** int8 si pierde ≤ 0.01 frente a float32.

In [ ]:
pq_v = {v: pq_S(EMB[v]["Q"] @ EMB[v]["D"].T)[I_DEV] for v in EMB}
r = paired_bootstrap(pq_v["v2"], pq_v["v1"])
modelo = "v2" if r["ci_low"] > 0 else "v1"
print(f"v2 − v1 en dev: Δ={r['diff']:+.3f} IC95=[{r['ci_low']:+.3f},{r['ci_high']:+.3f}] → modelo = {modelo}")
D_full, Q_full = EMB[modelo]["D"], EMB[modelo]["Q"]
dims = [d for d in (16, 32, 48, 64, 96, 128, 256, 384, 512, 768, 1024) if d <= D_full.shape[1]]
nd_dim = {d: pq_S(l2n(Q_full[:, :d]) @ l2n(D_full[:, :d]).T)[I_DEV].mean() for d in dims}
dim = min(d for d in dims if nd_dim[d] >= nd_dim[dims[-1]] - 0.01)
D_dim, Q_dim = l2n(D_full[:, :dim]), l2n(Q_full[:, :dim])
nd_i8 = pq_S(Q_dim @ int8_ida_vuelta(D_dim).T)[I_DEV].mean()
precision = "int8" if nd_i8 >= nd_dim[dim] - 0.01 else "float32"
D_sel = int8_ida_vuelta(D_dim) if precision == "int8" else D_dim
DECISIONES.update(modelo=VERS[modelo].clave, dim=dim, precision=precision)
METRICAS.update(dev_nd_dim=nd_dim, dev_nd_int8=nd_i8)
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.plot(list(nd_dim), list(nd_dim.values()), "o-", color=AZUL, label="float32")
ax.scatter([dim], [nd_i8], color=NARANJA, s=80, zorder=3, label=f"int8 a d={dim}")
ax.axhline(nd_dim[dims[-1]] - 0.01, color=GRIS, ls="--", label="tolerancia: completa − 0.01")
ax.set_xscale("log", base=2); ax.set_xlabel("dimensión (primeras d coordenadas, renormalizadas)"); ax.set_ylabel("nDCG@10 en dev")
ax.set_title(f"Dimensión y precisión del embedder ({modelo})"); ax.legend(fontsize=8); plt.tight_layout(); plt.show()
print(f"decisión: d={dim}, {precision} · bytes/vector = {dim * (1 if precision == 'int8' else 4)}")

> 👀 **Qué debes observar:** la curva sube rápido y se aplana: las últimas dimensiones aportan poco, así que la regla elige una dimensión menor que la completa con pérdida acotada, e int8 apenas cambia el nDCG. ⚠️ Con el fallback (SVD), recortar equivale a una PCA y funciona por construcción; con un modelo real **solo** es seguro si se entrenó con Matryoshka (módulo 12): mide siempre.

## 4. Fusión híbrida: RRF vs combinación lineal (en dev, con CV)
Regla: **RRF** por defecto; **lineal** solo si su estimación por validación cruzada en dev supera a RRF con un IC pareado que excluye el 0 (la simplicidad tiene valor: RRF no tiene pesos que se desajusten cuando cambie el embedder).

In [ ]:
bm_sel, tok_sel, S_lex = S_LEX[mejor_lex]
S_den = Q_dim @ D_sel.T
cands = {"solo BM25": S_lex, "solo denso": S_den, "RRF": rrf_mat(S_den, S_lex)}
alfas = np.linspace(0, 1, 21)
PQa = np.array([pq_S(lineal_lista(S_den, S_lex, a))[I_DEV] for a in alfas])   # misma normalización que el servicio
pl = np.array_split(np.random.default_rng(SEED).permutation(len(I_DEV)), 5); lin_cv = np.zeros(len(I_DEV))
for f in pl:
    tr = np.setdiff1d(np.arange(len(I_DEV)), f); lin_cv[f] = PQa[PQa[:, tr].mean(1).argmax(), f]
pq_dev = {k: pq_S(S)[I_DEV] for k, S in cands.items()}; pq_dev["lineal (CV)"] = lin_cv
r = paired_bootstrap(lin_cv, pq_dev["RRF"])
fusion = "lineal" if r["ci_low"] > 0 else "RRF"
ALFA = float(alfas[PQa.mean(1).argmax()])
DECISIONES.update(fusion=fusion, alfa_lineal=ALFA if fusion == "lineal" else None)
METRICAS["fusion_delta"] = r
fig, ax = plt.subplots(figsize=(7.5, 3.8))
nombres = list(pq_dev); vals = [pq_dev[k].mean() for k in nombres]; cis = [bootstrap_ci(pq_dev[k]) for k in nombres]
ax.bar(nombres, vals, color=[GRIS, AZUL, VERDE, NARANJA], yerr=[[v - c[1] for v, c in zip(vals, cis)], [c[2] - v for v, c in zip(vals, cis)]], capsize=4)
ax.set_ylabel("nDCG@10 en dev (IC95)"); ax.set_title("Fusión híbrida en dev"); ax.set_ylim(min(vals) - .15, 1.0); plt.tight_layout(); plt.show()
print(f"lineal(CV) − RRF: Δ={r['diff']:+.3f} IC95=[{r['ci_low']:+.3f},{r['ci_high']:+.3f}] → fusión = {fusion}")

> 👀 **Qué debes observar:** el híbrido mejora (o iguala) al mejor componente. La barra «lineal (CV)» es una estimación **honesta** (cada pliegue se puntúa con el α elegido en los otros cuatro) y usa la **misma normalización que el servicio** (min-max dentro del top-50 de cada lista): si esa estimación supera a RRF con un IC que excluye el 0, la regla elige lineal y el servicio la implementa con el α ajustado en todo dev; si el IC incluye el 0, RRF: no pagues complejidad por una mejora que no puedes demostrar. Con un embedder neuronal real, cuyos scores están mejor calibrados que los del fallback, el resultado de esta comparación puede cambiar: por eso es una regla y no una constante.

## 5. Índice: elegir con el árbol del módulo 19 y verificar recall
Con 400 documentos todo es trivial. Para tomar una decisión realista, ampliamos los vectores elegidos con **copias ruidosas** hasta $N$ = 20 000 (como el proyecto 14), medimos Flat vs HNSW y **proyectamos** memoria a 10 M de chunks.

In [ ]:
rng = np.random.default_rng(SEED)
N_SIM = 20_000 if FAST_DEV_RUN else 200_000
base = D_sel[rng.integers(0, len(D_sel), N_SIM)]
X = l2n(base + 0.35 * rng.normal(size=base.shape).astype(np.float32) / np.sqrt(dim))
Qx = Q_dim[I_DEV]
flat = faiss.IndexFlatIP(dim); flat.add(X); gt = flat.search(Qx, 10)[1]
def mide(idx, Q, params=None):
    t = []
    for q in Q:
        t0 = time.perf_counter(); idx.search(q[None], 10, params=params); t.append((time.perf_counter() - t0) * 1e3)
    return float(np.percentile(t, 50)), float(np.percentile(t, 99))
filas = [{"índice": "Flat", "ef": None, "recall@10": 1.0, **dict(zip(("p50_ms", "p99_ms"), mide(flat, Qx)))}]
hnsw = faiss.IndexHNSWFlat(dim, 32, faiss.METRIC_INNER_PRODUCT); hnsw.hnsw.efConstruction = 80; hnsw.add(X)
for ef in (16, 32, 64, 128):
    p = faiss.SearchParametersHNSW(efSearch=ef); got = hnsw.search(Qx, 10, params=p)[1]
    filas.append({"índice": "HNSW M=32", "ef": ef, "recall@10": float(np.mean([len(set(a) & set(b)) / 10 for a, b in zip(gt, got)])),
                  **dict(zip(("p50_ms", "p99_ms"), mide(hnsw, Qx, p)))})
df_ann = pd.DataFrame(filas)
ok = df_ann[(df_ann["índice"] != "Flat") & (df_ann["recall@10"] >= 0.95)]
fila_hnsw = ok.sort_values("p99_ms").iloc[0] if len(ok) else df_ann.iloc[-1]
bpv = dim * (1 if precision == "int8" else 4)
ram_10M = {"Flat": 1e7 * bpv / 1e9, "HNSW": 1e7 * (bpv + 256) / 1e9}
DECISIONES.update(indice=f"HNSW M=32 efSearch={int(fila_hnsw.ef)}", ef_search=int(fila_hnsw.ef))
METRICAS.update(recall_ann=float(fila_hnsw["recall@10"]), ram_10M_gb=ram_10M["HNSW"])
fig, ax = plt.subplots(figsize=(7.5, 4))
h = df_ann[df_ann["índice"] != "Flat"]
ax.plot(h["recall@10"], h["p99_ms"], "o-", color=AZUL, label="HNSW (ef = 16…128)")
for _, rr in h.iterrows():
    ax.annotate(f"ef={int(rr.ef)}", (rr["recall@10"], rr["p99_ms"]), xytext=(4, 4), textcoords="offset points", fontsize=8)
ax.scatter([1.0], [df_ann.iloc[0]["p99_ms"]], color=NARANJA, s=80, label="Flat (exacto)"); ax.axvline(0.95, color=ROJO, ls="--", label="recall ≥ 0.95")
ax.set_xlabel("neighbor recall@10"); ax.set_ylabel("p99 (ms, batch=1)"); ax.set_title(f"Índice: N={N_SIM:,}, d={dim}"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
print(f"RAM proyectada a 10 M ({precision}, d={dim}): Flat {ram_10M['Flat']:.1f} GB · HNSW {ram_10M['HNSW']:.1f} GB")
df_ann.round(3)

> 👀 **Qué debes observar:** HNSW alcanza el recall exigido con un `efSearch` moderado y su p99 casi no depende de $N$, mientras que Flat crece lineal (módulo 19, §1). Con 10 M de chunks, la dimensión y precisión elegidas en §3 deciden si el índice cabe en una réplica: por eso la decisión del embedder y la del índice **no** son independientes. En el servicio de Políglota (400 docs reales) usamos HNSW con el `efSearch` elegido para que la configuración probada sea la de producción.

## 6. El servicio integrado (módulo 18) bajo tráfico
Montamos `ServicioPoliglota` con **exactamente** las decisiones anteriores: analizador de BM25 (mismo tokenizador al indexar y al consultar: contrato F3 del módulo 19), embedder recortado a `dim`, vectores en la precisión elegida, HNSW con el `efSearch` elegido, la fusión elegida (RRF o lineal con su α, normalizada dentro de cada lista igual que en §4) y caché versionada.

> ### 🎬 GEMINI-VIDEO · ID: `M20-VID-01` · ⚠️ REEMPLAZAR ESTA CELDA
> **Recorrido de una consulta por Políglota v1.0**
>
> **Qué muestra:** Animación de una consulta real atravesando el sistema completo, con los números que el capstone mide en cada etapa.
> **Formato:** MP4 16:9, 1080p | Duración: 30-35 s | **Texto alternativo:** Animación de una consulta que pasa por normalización, caché, BM25 y denso en paralelo, fusión RRF y top-3, y una segunda consulta que acierta en caché.
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Video de diagrama técnico animado, 16:9, fondo blanco, paleta: azul #1F6FEB (consulta), naranja #F28E2B (BM25), verde #2CA02C (denso), gris #6B7280 (infraestructura). Sans-serif grande, sin voz en off, ritmo calmado.
> 0–4 s: aparece una tarjeta con la consulta "cómo invalidar la caché de sesiones" y el texto "Usuario → /search".
> 4–9 s: la consulta entra en la caja "Normalización NFC + minúsculas"; aparece la clave "versión v2-d96-int8 + texto normalizado" y una caja "Caché LRU": primero un letrero rojo "fallo", luego la consulta sigue.
> 9–15 s: la consulta se divide en dos flechas: naranja hacia "BM25 (palabras + identificadores)" y verde hacia "Embedder (d recortada) → HNSW efSearch". Cada flecha devuelve una lista de 5 documentos con su rango 1..5.
> 15–21 s: las dos listas se funden en la caja "RRF: Σ 1/(60 + rango)"; los documentos se reordenan visiblemente; un documento que estaba 3.º en ambas sube al 1.º.
> 21–26 s: aparece el top-3 final en una tarjeta; debajo, el texto "latencia registrada: p50 / p99" y "vector guardado en caché".
> 26–32 s: la misma consulta se repite y la caja "Caché LRU" muestra "acierto": la flecha verde salta directamente al índice. Texto final: "Cada etapa = una decisión medida en dev".
> Evita: cifras inventadas, logotipos, transiciones bruscas, texto pequeño.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M20-VID-01.mp4` en la carpeta del módulo y reemplaza esta celda por: `<video src="assets/M20-VID-01.mp4" controls width="100%"></video>`
>
> **Mientras tanto (placeholder):**
> ```text
> "cómo invalidar la caché de sesiones" → NFC/minúsculas → clave(versión+texto) → caché: FALLO
>    ├─ BM25 (palabras+ident) → rangos 1..k ─┐
>    └─ embedder[:d] → HNSW(ef) → rangos ────┴→ RRF Σ1/(60+r) → top-3 (+latencia, +caché)
> 2ª vez: caché ACIERTO → salta el embedder
> ```

> 👀 **Qué debes observar:** la segunda consulta se salta el embedder: eso es lo que mide el *hit rate*; y la fusión puede subir al primer puesto un documento que no era el primero en ninguna lista.

In [ ]:
VERSION = f"{modelo}-d{dim}-{precision}"
FUS = dict(fusion="lineal" if DECISIONES["fusion"] == "lineal" else "rrf", alfa=ALFA)
svc = ServicioPoliglota(docs, bm_sel, cache_cap=1024, tokenizador=tok_sel, **FUS)
svc.registrar_version(VERSION, EmbedderRecortado(VERS[modelo], dim), D_sel, indice="hnsw", ef_search=DECISIONES["ef_search"])
svc.alias["prod"] = VERSION
modo = "hibrido"
run_svc = {qids[i]: [d for d, _ in svc.buscar(q_texts[i], k=10, modo=modo)] for i in range(len(qids))}   # frío
for i in range(len(qids)):
    svc.buscar(q_texts[i], k=10, modo=modo)                                                                # caliente
lat = np.array(svc.lat_ms); n = len(qids)
def trafico_zipf(n, universo, s=1.1, p_variante=0.35, seed=SEED):
    rng = np.random.default_rng(seed); p = 1.0 / np.arange(1, len(universo) + 1) ** s; p /= p.sum(); out = []
    for i in rng.choice(len(universo), size=n, p=p):
        t = universo[i]
        out.append(rng.choice([t.upper(), "  " + t, t + " "]) if rng.random() < p_variante else t)
    return out
universo = [q_texts[i % len(q_texts)] + ("" if i < len(q_texts) else f" {i // len(q_texts)}") for i in range(3000)]
svc_z = ServicioPoliglota(docs, bm_sel, cache_cap=1024, tokenizador=tok_sel, **FUS)
svc_z.registrar_version(VERSION, EmbedderRecortado(VERS[modelo], dim), D_sel, indice="hnsw", ef_search=DECISIONES["ef_search"]); svc_z.alias["prod"] = VERSION
for t in trafico_zipf(5000, universo): svc_z.buscar(t)
METRICAS.update(p50_frio=float(np.percentile(lat[:n], 50)), p99_frio=float(np.percentile(lat[:n], 99)),
                p50_caliente=float(np.percentile(lat[n:], 50)), p99_caliente=float(np.percentile(lat[n:], 99)),
                p99_zipf=float(np.percentile(svc_z.lat_ms, 99)), hit_rate=svc_z.cache.hit_rate)
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.hist(lat[:n], bins=30, color=NARANJA, alpha=.7, label="frío (caché vacía)"); ax.hist(lat[n:], bins=30, color=AZUL, alpha=.7, label="caliente")
ax.hist(svc_z.lat_ms, bins=30, color=VERDE, alpha=.5, label=f"tráfico Zipf (hit rate {svc_z.cache.hit_rate:.2f})")
ax.set_xlabel("latencia por query (ms, CPU, batch=1)"); ax.set_ylabel("nº de queries"); ax.set_yscale("log"); ax.set_title("Latencia del servicio Políglota v1.0")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()
print({k: round(v, 3) for k, v in METRICAS.items() if k.startswith(("p5", "p9", "hit"))})

> 👀 **Qué debes observar:** la distribución fría es la que ve un usuario con una query nueva y es la que cuenta para el criterio de p99; el tráfico Zipf mezcla aciertos (rápidos) y fallos (lentos), y su cola la ponen los fallos. En producción con un embedder neuronal, el embedder domina la latencia fría: ahí es donde pagan el micro-batching y la GPU (módulo 18).

## 6b. Presupuesto de error: ¿qué aproximación del despliegue te cuesta calidad?
Entre el «denso exacto en float32 con $d$ completa» de un paper y lo que sirves hay una cadena de aproximaciones: recortar $d$, cuantizar, un índice ANN, la fusión. Un equipo de búsqueda serio **descompone** la diferencia por escalón (en dev, una variable cada vez) para saber dónde gastar: si el ANN no cuesta nada, subir `efSearch` es tirar latencia; si cuesta, es lo primero que se arregla.

La segunda parte separa dos cosas que se confunden a menudo: el **neighbor recall@10** del ANN (cuántos vecinos del exacto devuelve, relevantes o no; módulo 14) y lo que el usuario nota: cuántos de los **documentos relevantes** que encontraba el exacto se pierden y cuánto nDCG cuesta. Como con 400 documentos el HNSW es prácticamente exacto, añadimos al índice las $N$ copias ruidosas del §5 como **distractores no relevantes** (casi duplicados: el caso difícil) y usamos un grafo pequeño (M = 16) para que haya fallos visibles.

In [ ]:
esc = {"exacto · d completa · float32": pq_S(Q_full @ D_full.T)[I_DEV].mean(),
       f"+ d recortada ({dim})": pq_S(Q_dim @ D_dim.T)[I_DEV].mean(),
       f"+ {precision}": pq_S(Q_dim @ D_sel.T)[I_DEV].mean()}
run_den = {qids[i]: [d for d, _ in svc.buscar(q_texts[i], k=10, modo="denso")] for i in I_DEV}
esc["+ HNSW del servicio"] = ndcg_por_query(run_den)[I_DEV].mean()
esc["+ fusión (servicio completo)"] = ndcg_por_query(run_svc)[I_DEV].mean()
pasos = list(esc); delta = [0.0] + [esc[b] - esc[a] for a, b in zip(pasos, pasos[1:])]
print("\n".join(f"{p:<32} nDCG@10 dev = {esc[p]:.3f}  (Δ escalón {d:+.3f})" for p, d in zip(pasos, delta)))
# ANN: neighbor recall frente a recall de los relevantes, con distractores casi duplicados
Xd = np.vstack([l2n(D_sel), X]).astype(np.float32); ids_d = doc_ids + [f"distractor_{j}" for j in range(len(X))]
Qd = Q_dim[I_DEV]; gt_d = faiss.IndexFlatIP(dim); gt_d.add(Xd); gx = gt_d.search(Qd, 10)[1]
hn = faiss.IndexHNSWFlat(dim, 16, faiss.METRIC_INNER_PRODUCT); hn.hnsw.efConstruction = 40; hn.add(Xd)
nd_ids = lambda I: ndcg_por_query({qids[i]: [ids_d[j] for j in fila] for i, fila in zip(I_DEV, I)})[I_DEV]
nd_ex = nd_ids(gx); filas_b = []
for ef in (10, 16, 32, 64, 128):
    I = hn.search(Qd, 10, params=faiss.SearchParametersHNSW(efSearch=ef))[1]
    rel_ex = [{ids_d[j] for j in a} & set(qrels[qids[i]]) for i, a in zip(I_DEV, gx)]
    rel_hn = [{ids_d[j] for j in b} for b in I]
    filas_b.append({"ef": ef, "neighbor_recall@10": np.mean([len(set(a) & set(b)) / 10 for a, b in zip(gx, I)]),
                    "recall_relevantes": sum(len(r & h) for r, h in zip(rel_ex, rel_hn)) / max(sum(map(len, rel_ex)), 1),
                    "pérdida_nDCG": nd_ex.mean() - nd_ids(I).mean()})
df_b = pd.DataFrame(filas_b)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
a1.barh(pasos[::-1], [esc[p] for p in pasos[::-1]], color=[VERDE, AZUL, AZUL, AZUL, GRIS])
a1.set_xlim(min(esc.values()) - .1, max(esc.values()) + .05); a1.set_xlabel("nDCG@10 en dev"); a1.set_title("Escalones del despliegue")
a2.plot(df_b.ef, df_b["neighbor_recall@10"], "o-", color=AZUL, label="neighbor recall@10 (vs exacto)")
a2.plot(df_b.ef, df_b["recall_relevantes"], "s-", color=VERDE, label="relevantes del exacto que se conservan")
a2.plot(df_b.ef, 1 - df_b["pérdida_nDCG"], "^--", color=NARANJA, label="1 − pérdida de nDCG@10")
a2.set_xscale("log", base=2); a2.set_xlabel("efSearch"); a2.set_title(f"HNSW M=16 con {len(X):,} distractores"); a2.legend(fontsize=8)
plt.tight_layout(); plt.show()
METRICAS["presupuesto_error"] = dict(zip(pasos, delta))
df_b.round(3)

> 👀 **Qué debes observar:** (1) en la barra de escalones, el salto grande está en la **fusión** (y, con modelos reales, en el **modelo**), no en recortar $d$, en int8 ni en el ANN: por eso el §3 recorta con tolerancia y el §5 no persigue recall 1.0. (2) A la derecha, con `efSearch` bajo el neighbor recall cae mucho más que los relevantes conservados y que el nDCG (en la ejecución de referencia, a ef = 10: neighbor recall ≈ 0.57, relevantes ≈ 0.77, pérdida de nDCG ≈ 0.03): los vecinos que el ANN se salta suelen ser distractores o estar al fondo del top-10. Con ef alto la «pérdida» puede salir **negativa**: el exacto llena el top-10 de casi duplicados y el ANN, al saltarse alguno, deja entrar un relevante. No es una razón para bajar `efSearch`; es la prueba de que el neighbor recall mide **fidelidad al exacto**, no calidad. El criterio recall ≥ 0.95 es una cota prudente y barata de medir sin qrels; un sistema maduro decide `efSearch` mirando también la pérdida con qrels **por slice** (si los relevantes de un idioma viven en una región peor conectada del grafo, ese slice pierde más que la media).

## 7. Evaluación final: test ciego, una sola vez
Comparamos el **servicio** (no un notebook paralelo: se evalúa lo que se despliega) contra el baseline BM25 del módulo 00 y contra el denso solo, en las queries de **test** que no se usaron para ninguna decisión.

In [ ]:
pq_final = ndcg_por_query(run_svc)[I_TEST]
pq_bm25_00 = pq_S(S_BM25)[I_TEST]
pq_denso = pq_S(S_den)[I_TEST]
comp = {"BM25 (módulo 00)": pq_bm25_00, "denso solo": pq_denso, "Políglota v1.0 (servicio)": pq_final}
r_bm = paired_bootstrap(pq_final, pq_bm25_00); r_de = paired_bootstrap(pq_final, pq_denso)
lt = q_lang[I_TEST]
slices = {l: (pq_final[lt == l].mean(), pq_bm25_00[lt == l].mean()) for l in ("es", "en", "code")}
METRICAS.update(test_final=pq_final.mean(), test_bm25=pq_bm25_00.mean(), test_denso=pq_denso.mean(), delta_bm25=r_bm, delta_denso=r_de, slices=slices)
fig, ax = plt.subplots(figsize=(9, 4)); x = np.arange(4); w = .27
for j, (k, v) in enumerate(comp.items()):
    vals = [v.mean()] + [v[lt == l].mean() for l in ("es", "en", "code")]
    ci = bootstrap_ci(v)
    ax.bar(x + (j - 1) * w, vals, w, color=[GRIS, AZUL, VERDE][j], label=k)
    ax.errorbar(x[0] + (j - 1) * w, vals[0], yerr=[[vals[0] - ci[1]], [ci[2] - vals[0]]], color="k", capsize=3)
ax.set_xticks(x); ax.set_xticklabels(["global (IC95)", "es", "en", "code"]); ax.set_ylabel("nDCG@10 en test ciego"); ax.set_ylim(0, 1.05)
ax.set_title("Políglota v1.0 vs baselines (test ciego)"); ax.legend(fontsize=8); plt.tight_layout(); plt.show()
for nombre, rr in (("vs BM25", r_bm), ("vs denso", r_de)):
    print(f"Políglota {nombre}: Δ={rr['diff']:+.3f} IC95=[{rr['ci_low']:+.3f},{rr['ci_high']:+.3f}] p={rr['p_value']:.3f}")

> 👀 **Qué debes observar:** con ~55 queries de test los IC son anchos: una mejora de pocos puntos sobre BM25 puede no ser significativa, y eso **se reporta tal cual** (el criterio de aceptación exige no ser peor, con margen). El desglose por idioma enseña dónde está el valor del híbrido (típicamente código, gracias a los identificadores y al denso) y si algún slice empeora. Con datos reales y un embedder neuronal multilingüe, espera diferencias mayores en paráfrasis y cruce ES↔EN (módulos 13 y 19).

## 8. Informe técnico y criterios de aceptación (generados desde las métricas)
El informe se **genera** desde `DECISIONES` y `METRICAS`: si cambias una decisión y re-ejecutas, el informe se actualiza solo (nada de copiar números a mano). Se guarda como `poliglota_informe.md` junto a `poliglota_config.json`.

> ### 🎨 GEMINI-IMAGEN · ID: `M20-IMG-02` · ⚠️ REEMPLAZAR ESTA CELDA
> **Anatomía de un informe técnico de decisiones**
>
> **Qué muestra:** La estructura de una página del informe que genera el capstone: decisiones con evidencia, resultados en test ciego, costes/latencia y tabla de criterios de aceptación.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Maqueta de una página de informe con decisiones, resultados en test, costes y latencia, criterios de aceptación y limitaciones.
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Maqueta de documento técnico de una página, vista frontal, fondo blanco, paleta: azul #1F6FEB (títulos), verde #2CA02C (criterio cumplido), naranja #F28E2B (criterio no cumplido), gris #6B7280 (texto de relleno). Sans-serif legible.
> Título arriba: "Políglota v1.0 · Informe técnico de decisiones". Debajo, cinco bloques apilados con encabezados azules:
> 1 "Decisiones (tomadas en dev)": tabla de 3 columnas "Componente | Decisión | Evidencia" con filas "Tokenizer", "Analizador BM25", "Embedder", "Fusión", "Índice" (el contenido de las celdas en líneas grises, sin números).
> 2 "Resultados en test ciego": una barra horizontal con tres segmentos rotulados "BM25", "denso", "Políglota" y un intervalo de confianza dibujado como bigote.
> 3 "Costes y latencia": tres pastillas "re-embedding $", "RAM GB/réplica", "p50 / p99 ms".
> 4 "Criterios de aceptación": tabla "Criterio | Umbral | Valor | OK" con cinco filas: cuatro con un ✓ verde y una con un ✗ naranja.
> 5 "Limitaciones y siguiente iteración": tres viñetas grises.
> Al margen derecho, una flecha gris vertical con el texto "generado desde las métricas, no escrito a mano".
> Evita: números concretos, logotipos, texto ilegible, sombras decorativas.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M20-IMG-02.png` en la carpeta del módulo y reemplaza esta celda por: `![Maqueta de una página de informe con decisiones, resultados en test, costes y latencia, criterios de aceptación y limitaciones.](assets/M20-IMG-02.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> # Políglota v1.0 · Informe técnico
> [Decisiones (dev)]  Componente | Decisión | Evidencia
> [Test ciego]        BM25 | denso | Políglota (IC95)
> [Costes/latencia]   $ re-embedding · GB/réplica · p50/p99
> [Aceptación]        Criterio | Umbral | Valor | ✓/✗
> [Limitaciones]      • • •        ← generado desde METRICAS/DECISIONES
> ```

> 👀 **Qué debes observar:** cada bloque responde a una pregunta de la CTO (¿qué elegiste?, ¿es mejor?, ¿cuánto cuesta?, ¿cumple?, ¿qué falta?). Si tu informe no contesta alguna, está incompleto aunque los números sean buenos.

In [ ]:
coste_reembed = 1e7 * 400 / 1e6 * 0.02
checks = {
    "calidad_vs_bm25": METRICAS["test_final"] >= METRICAS["test_bm25"] - 1e-12 and METRICAS["delta_bm25"]["ci_low"] > -0.03,
    "peor_slice": all(f >= b - 0.05 for f, b in METRICAS["slices"].values()),
    "recall_ann": METRICAS["recall_ann"] >= 0.95,
    "latencia_p99_ms": METRICAS["p99_frio"] <= 50,
    "ram_10M_gb": METRICAS["ram_10M_gb"] <= 16,
    "coste_reembed_usd": coste_reembed <= 500,
    "hit_rate": METRICAS["hit_rate"] >= 0.70,
}
valores = {"calidad_vs_bm25": f"{METRICAS['test_final']:.3f} vs {METRICAS['test_bm25']:.3f} (IC Δ ≥ {METRICAS['delta_bm25']['ci_low']:+.3f})",
           "peor_slice": ", ".join(f"{l}: {f:.3f}/{b:.3f}" for l, (f, b) in METRICAS["slices"].items()),
           "recall_ann": f"{METRICAS['recall_ann']:.3f}", "latencia_p99_ms": f"{METRICAS['p99_frio']:.1f} ms (frío)",
           "ram_10M_gb": f"{METRICAS['ram_10M_gb']:.1f} GB", "coste_reembed_usd": f"{coste_reembed:.0f} $", "hit_rate": f"{METRICAS['hit_rate']:.2f}"}
tabla = "\n".join(f"| {k} | {ACEPTACION[k]} | {valores[k]} | {'✅' if ok else '❌'} |" for k, ok in checks.items())
informe = f"""# Políglota v1.0 · Informe técnico de decisiones
**Fuente de datos:** {corpus['source']} · **docs:** {len(docs)} · **queries:** dev {len(DEV)} / test ciego {len(TEST)} · **semilla:** {SEED}

## Decisiones (tomadas en dev)
| Componente | Decisión | Evidencia |
|---|---|---|
| Tokenizer (coste) | {DECISIONES['tokenizer_costo']} | fertilidad code/es ≈ {df_tok[(df_tok.vocab == 4000) & (df_tok.lang == 'code')].fertilidad.item() / df_tok[(df_tok.vocab == 4000) & (df_tok.lang == 'es')].fertilidad.item():.1f}× |
| Analizador BM25 | {DECISIONES['analizador_bm25']} | mejor nDCG@10 en dev entre 3 analizadores |
| Embedder | {DECISIONES['modelo']} · d={DECISIONES['dim']} · {DECISIONES['precision']} | menor d a ≤ 0.01 de la completa; int8 pierde {METRICAS['dev_nd_dim'][dim] - METRICAS['dev_nd_int8']:+.3f} |
| Fusión | {DECISIONES['fusion']}{f" (α={ALFA:.2f})" if DECISIONES['fusion'] == 'lineal' else ''} | lineal(CV) − RRF en dev: Δ={METRICAS['fusion_delta']['diff']:+.3f}, IC95 [{METRICAS['fusion_delta']['ci_low']:+.3f}, {METRICAS['fusion_delta']['ci_high']:+.3f}] |
| Índice | {DECISIONES['indice']} | recall@10 = {METRICAS['recall_ann']:.3f} con N={N_SIM:,} |
| Versión servida | `{VERSION}` | alias `prod` → `{svc.alias['prod']}` |

## Resultados en test ciego
nDCG@10: **Políglota {METRICAS['test_final']:.3f}** · BM25 {METRICAS['test_bm25']:.3f} · denso {METRICAS['test_denso']:.3f}.
Δ vs BM25 = {METRICAS['delta_bm25']['diff']:+.3f} (IC95 [{METRICAS['delta_bm25']['ci_low']:+.3f}, {METRICAS['delta_bm25']['ci_high']:+.3f}], p = {METRICAS['delta_bm25']['p_value']:.3f}).

## Costes y latencia (proyección a 10 M chunks × 400 tokens)
Re-embedding: {coste_reembed:.0f} $ con API a 0.02 $/1M (oct-2026) · RAM del índice: {METRICAS['ram_10M_gb']:.1f} GB/réplica · latencia (CPU, batch=1): p50 {METRICAS['p50_frio']:.1f} ms / p99 {METRICAS['p99_frio']:.1f} ms en frío; hit rate Zipf {METRICAS['hit_rate']:.2f}.

## Criterios de aceptación
| Criterio | Umbral | Valor | OK |
|---|---|---|---|
{tabla}

## Limitaciones y siguiente iteración
* Golden set pequeño ({len(TEST)} queries de test): IC anchos. Ampliar con queries reales muestreadas por slice.
* {'Corpus SINTÉTICO y embedders fallback: repetir con datos y modelos reales antes de decidir en producción.' if corpus['source'] == 'synthetic' else 'Datos reales: validar online con interleaving (módulo 18).'}
* Multimodal (módulo 16) fuera de esta versión: añadir como tercer retriever en RRF y medir en un slice de capturas.
"""
open("poliglota_informe.md", "w", encoding="utf-8").write(informe)
json.dump({"version": VERSION, **{k: (v if not isinstance(v, (np.floating, np.integer)) else v.item()) for k, v in DECISIONES.items()}},
          open("poliglota_config.json", "w"), indent=1, ensure_ascii=False)
from IPython.display import Markdown, display
display(Markdown(informe))
print(f"Criterios cumplidos: {sum(checks.values())}/{len(checks)}")

> 👀 **Qué debes observar:** el informe dice **qué se decidió, con qué evidencia y qué no se cumple**, y lo dice con los números de esta ejecución. Un criterio en ❌ no es un fracaso del informe: es la información que necesita la CTO (por ejemplo, un p99 en CPU que en GPU sí se cumpliría, o un slice que exige más datos).

## 9. (Opcional) Extensión multimodal
Para buscar en capturas y PDFs (módulo 16) sin rehacer el sistema:
1. **Primera etapa sin cambios**: OCR → texto → mismo pipeline (BM25 + denso). Es la opción barata y la que el árbol del módulo 19 elige cuando el índice multi-vector no cabe.
2. **Tercer retriever**: un embedder visual (CLIP/SigLIP para fotos, ColPali para páginas) produce su lista de rangos, que entra en la **misma RRF** (RRF no necesita calibrar escalas entre modalidades).
3. **Segunda etapa**: MaxSim visual solo sobre el top-K de páginas (vectores multi-vector en SSD).
4. **Evaluación**: un slice nuevo «capturas» en el golden set; el criterio de aceptación se amplía a ese slice.
El proyecto 16 (*Políglota Visual*) ya contiene el estudiante visual y MaxSim sobre patches: su salida (`S_visual`, matriz queries × páginas) se fusiona con `rrf_mat(S_den, S_lex, S_visual)`.

## 🗺️ Mapa de madurez de las piezas de Políglota v1.0 (a oct-2026)
Cierra el sistema mirando la madurez de cada pieza que acabas de integrar: casi todo es 🟢, y lo 🟡 es justo lo que el informe debe señalar como riesgo.

| Pieza | Elección de referencia | Madurez | Fuente / módulo |
|---|---|---|---|
| Normalización | NFC en indexación y consulta | 🟢 | UAX #15 · M01 |
| Tokenizer (coste y contexto) | BPE byte-level (HF `tokenizers`) | 🟢 | Sennrich et al. 2016; Wang et al. 2019 · M02–M04 |
| Analizador léxico de BM25 | palabras + partes de identificadores | 🟢 | Robertson & Zaragoza 2009 · M00, M13 |
| Embedder | bi-encoder multilingüe (e5) o fallback; dimensión recortada (Matryoshka) | 🟢 / 🟡 (Matryoshka en modelos sin entrenarla) | Wang et al. 2024; Kusupati et al. 2022 · M10–M12 |
| Precisión del índice | int8 calibrado | 🟢 | Shakir et al. 2024 · M12 |
| Fusión | RRF (lineal solo si gana con IC) | 🟢 | Cormack et al. 2009; Bruch et al. 2022 · M13 |
| Índice | Flat o HNSW según el árbol de M19 | 🟢 | Malkov & Yashunin 2016; Douze et al. 2024 · M14 |
| Evaluación | test ciego, bootstrap pareado, slices | 🟢 | Smucker et al. 2007 · M15 |
| Multimodal (opcional) | OCR + texto; ColPali en 2ª etapa | 🟡 | Faysse et al. 2024 · M16 |
| Servicio | caché versionada + alias + FastAPI | 🟢 | M18 |

## 🏭 En producción
* El patrón que has integrado — normalización simétrica, BM25 + denso con RRF, reranking opcional, índice HNSW/IVF cuantizado, caché versionada y alias de colección — es el que documentan los motores de búsqueda y vector DBs de uso general (Elasticsearch/OpenSearch, Vespa, Qdrant, Weaviate, pgvector) en sus guías de búsqueda híbrida (consultadas oct-2026).
* **Contextual Retrieval** (Anthropic, 2024) es un ejemplo publicado de mejora sobre este mismo esqueleto: un LLM añade a cada chunk un contexto de ≈ 50–100 tokens antes de embeberlo e indexarlo con BM25. En su evaluación, la tasa de fallo en el top-20 bajó de 5.7 % a 3.7 % (solo embeddings contextuales), 2.9 % (+ BM25 contextual) y 1.9 % (+ reranking), con un coste único estimado de 1.02 $ por millón de tokens de documento gracias al *prompt caching*. Fíjate en cómo lo reportan: una ablación por escalón, como el §6b, y el coste al lado de la ganancia.
* Los *technical reports* de embedders recientes (Qwen3 Embedding, EmbeddingGemma, Gemini Embedding; módulo 12) reportan por tarea e idioma: tu informe debe hacer lo mismo con tus slices.
* Decidir en dev, evaluar en test ciego, confirmar online con interleaving y A/B, y migrar con alias (módulos 15, 18, 19) es el ciclo de vida que se repite en cada versión.

## 🧠 Secretos de la élite
1. **Escribe los criterios de aceptación antes del primer experimento** y no los cambies después sin dejar constancia en el informe.
2. **Evalúa lo que despliegas y descompón la diferencia**: el número final sale del servicio (con su caché, su índice ANN y su cuantización), no de un notebook con fuerza bruta en float32; y la distancia entre ambos se reparte por escalón (§6b) para saber qué perilla tocar. El *neighbor recall* del ANN es una cota prudente, no la pérdida de calidad: mídelas las dos.
3. **Las decisiones interactúan**: dimensión y precisión del embedder fijan la memoria, que fija el índice, que fija la latencia; decide en ese orden y re-verifica al final.
4. **Informe generado, no escrito a mano**: cada cifra del informe sale de una variable; así no hay números obsoletos tras re-ejecutar.
5. **La regla «lo simple salvo prueba en contra»** (RRF salvo que lineal gane con IC) protege contra el *winner's curse* del módulo 19.
6. **El tokenizer del LLM no es el analizador de tu BM25**: úsalo para costes y contexto; el analizador léxico se elige por nDCG.
7. **Un criterio en ❌ con explicación vale más que diez en ✅ con umbrales ajustados a posteriori.**
8. **El golden set caduca el día del lanzamiento**: lo escribiste tú, con tu vocabulario y tus slices. La primera iteración tras desplegar es muestrear queries reales (sobre todo las huérfanas y las de clics bajos, módulo 18 §9), etiquetarlas y re-medir; la mezcla real de idiomas cambia qué sistema gana (gap offline/online, módulo 19 §3).

## ⚠️ Errores comunes
* Elegir la dimensión, la fusión o el `efSearch` mirando el test.
* Construir BM25 con un analizador e invocarlo con otro (contrato roto, fallo silencioso).
* Evaluar en float32 exacto y desplegar int8 + HNSW sin re-medir.
* Reportar solo la media global y esconder el slice que empeora.
* Proyectar memoria con `d` completa cuando el servicio usa `d` recortada (o al revés).
* Olvidar la versión del vector en la clave de caché al cambiar `dim` o `precision` (¡también son versión!).

## 📝 Autoevaluación
<details><summary><b>1 · (Cálculo)</b> Con d = 96 en int8 y HNSW (grafo ≈ 256 B/vector), ¿cuánta RAM ocupan 10 M de chunks? ¿Y con d = 768 float32?</summary>

$10^7 \times (96 + 256)$ B = **3.5 GB**; $10^7 \times (3072 + 256)$ B = **33.3 GB**. La elección del §3 multiplica por ~10 la capacidad por réplica.
</details>
<details><summary><b>2 · (Diagnóstico)</b> En dev el sistema gana +0.05 a BM25, en test +0.005 con IC que incluye el 0. ¿Qué concluyes y qué no?</summary>

Que no hay evidencia de mejora en test (probable optimismo de selección en dev, o un dev poco representativo). No concluyes que «es peor». Acciones: más queries de test por slice, revisar si alguna decisión se tomó con información del test, y validar online con interleaving.
</details>
<details><summary><b>3 · (Diagnóstico)</b> Cambias `dim` de 96 a 64 y redepliegas; los resultados de las queries populares son basura. ¿Por qué?</summary>

La versión servida debe incluir `dim` y `precision`; si la clave de caché o el alias no cambian, la caché sirve vectores de 96 d (o falla por dimensión, el caso «bueno»). `VERSION = f"{modelo}-d{dim}-{precision}"` existe por esto.
</details>
<details><summary><b>4 · (Transferencia)</b> Añades un slice de capturas (multimodal). ¿Qué cambia en los criterios de aceptación y en el informe?</summary>

Un criterio por slice (no peor que OCR+texto en capturas, con IC), el coste de almacenamiento multi-vector (en SSD) y la latencia de la segunda etapa MaxSim en el presupuesto de p99; el informe añade la decisión «OCR vs visual vs ambos» con su evidencia (módulo 16).
</details>
<details><summary><b>5 · (Transferencia)</b> La CTO pide pasar a GPU y a 100 M de chunks. ¿Qué secciones re-ejecutas y por qué?</summary>

§1 (umbrales de latencia en GPU), §3 (con 100 M la memoria puede forzar menos `d` o binario + rescore), §5 (índice: quizá IVF-PQ/DiskANN), §6 (latencia y micro-batching) y §8. El tokenizer y la fusión no cambian salvo que cambie el corpus.
</details>

## 📚 Referencias
* Sennrich, Haddow, Birch (2016). *Neural Machine Translation of Rare Words with Subword Units*. ACL 2016. https://arxiv.org/abs/1508.07909 · Wang, Cho, Gu (2019). *Neural Machine Translation with Byte-Level Subwords*. AAAI 2020. https://arxiv.org/abs/1909.03341
* Robertson & Zaragoza (2009). *The Probabilistic Relevance Framework: BM25 and Beyond*. Foundations and Trends in IR 3(4).
* Wang et al. (2024). *Multilingual E5 Text Embeddings: A Technical Report*. https://arxiv.org/abs/2402.05672 · Kusupati et al. (2022). *Matryoshka Representation Learning*. NeurIPS 2022. https://arxiv.org/abs/2205.13147
* Cormack, Clarke, Büttcher (2009). *Reciprocal Rank Fusion outperforms Condorcet and individual Rank Learning Methods*. SIGIR 2009 · Bruch, Gai, Ingber (2022). *An Analysis of Fusion Functions for Hybrid Retrieval*. ACM TOIS 42(1), 2023. https://arxiv.org/abs/2210.11934
* Malkov & Yashunin (2016). *Efficient and robust approximate nearest neighbor search using Hierarchical Navigable Small World graphs*. IEEE TPAMI 42(4), 2020. https://arxiv.org/abs/1603.09320 · Douze et al. (2024). *The Faiss library*. Preprint. https://arxiv.org/abs/2401.08281
* Smucker, Allan, Carterette (2007). *A Comparison of Statistical Significance Tests for Information Retrieval Evaluation*. CIKM 2007 · Thakur et al. (2021). *BEIR: A Heterogeneous Benchmark for Zero-shot Evaluation of Information Retrieval Models*. NeurIPS 2021 (Datasets & Benchmarks). https://arxiv.org/abs/2104.08663
* Faysse et al. (2024). *ColPali: Efficient Document Retrieval with Vision Language Models*. ICLR 2025. https://arxiv.org/abs/2407.01449
* Shakir, Aarsen, Lee (2024). *Binary and Scalar Embedding Quantization for Significantly Faster & Cheaper Retrieval* (blog de Hugging Face). https://huggingface.co/blog/embedding-quantization
* Anthropic (19-sep-2024). *Introducing Contextual Retrieval*. https://www.anthropic.com/news/contextual-retrieval
* Hugging Face `tokenizers`: https://github.com/huggingface/tokenizers · FAISS: https://github.com/facebookresearch/faiss
* Material del curso: `extras/GUIA_ENTREVISTAS.md` y `extras/LECTURA_PAPERS.md`.